
# LLM Training Infrastructure in PyTorch for Interview Prep
## DDP, FSDP, AMP, Activation Checkpointing, and Pipeline-Parallel Intuition

This notebook is a **teaching-first, interview-focused** walkthrough of the **main PyTorch training-infrastructure ideas** that show up in LLM training and systems interviews.

It is designed for:
- LLM training engineer interviews
- LLM inference/training infrastructure interviews
- ML systems interviews
- candidates who want **intuition + working code**

## What this notebook covers

By the end, you will have:
1. a tiny decoder-only language model used as a training toy
2. a baseline single-device training loop
3. a **DDP intuition demo** showing why gradient averaging matches a larger batch
4. a **FSDP intuition demo** showing why sharding reduces memory pressure
5. an **AMP demo** showing how `autocast` changes compute dtype
6. an **activation checkpointing demo** showing recomputation in backward
7. a **pipeline-parallel intuition demo** using microbatches and stages
8. interview-style exercises with **hints and answers**

## Philosophy

This notebook optimizes for:
- **clear mental models**
- **shape and memory intuition**
- **heavily commented code**
- **interview usefulness**

It does **not** try to be the fastest or most production-optimized implementation.

For production, you would usually also need:
- robust process launch and fault handling
- NCCL / Gloo / network debugging
- sharded checkpoint save/load
- real cluster schedulers
- observability and profiler workflows



## Why this matters for infrastructure roles

For infra-oriented LLM roles, it is not enough to know only the model architecture.

You usually also need to understand:
- how training is parallelized across processes and devices
- why memory becomes the bottleneck at large scale
- how mixed precision trades numerical range for performance
- why checkpointing trades extra compute for lower activation memory
- how pipeline stages and microbatches increase utilization

This notebook gives you a **strong first-principles foundation** for those topics.



## Practical note about Google Colab

Google Colab usually gives you:
- CPU, or
- one GPU

That means:
- the **intuition demos** in this notebook are fully runnable
- the **real multi-rank DDP / FSDP / pipeline launch snippets** are included as reference
- but true multi-process, multi-device training usually needs a larger environment than a typical single Colab notebook

That is normal.  
For interview prep, learning the concepts and seeing small working demos is already very valuable.


In [1]:

# ============================================================
# Imports and device setup
# ============================================================
# The notebook is written so that it can run on:
# - CPU
# - a single Colab GPU
#
# We will use the same model code in both cases.
# The main difference is where the tensors and modules live.

import copy
import math
import random
import time
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F

# Reproducibility matters for debugging and interview prep.
SEED = 7
random.seed(SEED)
torch.manual_seed(SEED)

# This can improve some matrix multiplication decisions on newer PyTorch builds.
if hasattr(torch, "set_float32_matmul_precision"):
    torch.set_float32_matmul_precision("high")

# Select GPU if available, otherwise CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"PyTorch version: {torch.__version__}")
print(f"Using device: {device}")

if torch.cuda.is_available():
    print(f"CUDA device name: {torch.cuda.get_device_name(0)}")
else:
    print("CUDA is not available. The notebook still works on CPU.")


ModuleNotFoundError: No module named 'matplotlib'


## 1. A tiny decoder-only language model

We need a model to hang the infrastructure ideas on.

We use a **small decoder-only language model** because:
- it feels more LLM-like than a plain MLP
- it stays small enough to run quickly
- the infrastructure ideas are easier to connect to a realistic model

We will train it on a toy next-token task:
- counting sequences like `<bos> 3 4 5 6 <eos>`
- input is everything except the last token
- target is everything except the first token

This is not about dataset difficulty.
This is about learning the infrastructure around the training loop.


In [ ]:

# ============================================================
# Tiny vocabulary and toy data
# ============================================================
# Token IDs:
# 0 -> padding
# 1 -> beginning of sequence
# 2 -> end of sequence
# 3..12 -> digits 0..9

PAD_ID = 0
BOS_ID = 1
EOS_ID = 2
DIGIT_OFFSET = 3
VOCAB_SIZE = 13

ID_TO_TOKEN: Dict[int, str] = {
    PAD_ID: "<pad>",
    BOS_ID: "<bos>",
    EOS_ID: "<eos>",
}

for digit in range(10):
    ID_TO_TOKEN[DIGIT_OFFSET + digit] = str(digit)

def token_ids_to_text(token_ids: List[int]) -> str:
    """Convert token IDs into readable text for debugging."""
    pieces: List[str] = []
    for token_id in token_ids:
        token = ID_TO_TOKEN.get(int(token_id), f"<?{int(token_id)}>")
        if token != "<pad>":
            pieces.append(token)
    return " ".join(pieces)

def make_counting_batch(
    batch_size: int,
    min_digits: int = 4,
    max_digits: int = 7,
    device: torch.device = torch.device("cpu"),
) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Build a batch of next-token training examples.

    Each full sequence looks like:
        <bos> d d+1 d+2 ... <eos>

    For language-model training:
        input_ids  = full[:, :-1]
        target_ids = full[:, 1:]
    """
    rows: List[List[int]] = []
    max_len = 0

    for _ in range(batch_size):
        # Randomize both the starting digit and the sequence length
        # so the model sees some variety.
        start_digit = random.randint(0, 9)
        seq_len = random.randint(min_digits, max_digits)

        # Build a simple counting pattern modulo 10.
        digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]

        # Full sequence includes BOS at the front and EOS at the end.
        full_sequence = [BOS_ID] + digits + [EOS_ID]
        rows.append(full_sequence)

        max_len = max(max_len, len(full_sequence))

    # Pad all rows to the same sequence length so they can share one tensor.
    padded_rows = [
        row + [PAD_ID] * (max_len - len(row))
        for row in rows
    ]

    full = torch.tensor(padded_rows, dtype=torch.long, device=device)

    # Next-token prediction:
    # - model sees all but the last token
    # - target is all but the first token
    input_ids = full[:, :-1]
    target_ids = full[:, 1:]

    return input_ids, target_ids

# Preview one batch.
preview_inputs, preview_targets = make_counting_batch(batch_size=3, device=device)

print("Readable sample 0")
print("input  ->", token_ids_to_text(preview_inputs[0].tolist()))
print("target ->", token_ids_to_text(preview_targets[0].tolist()))


In [ ]:

# ============================================================
# Tiny decoder-only language model
# ============================================================
# This is deliberately small and readable.
# It is not a production LLM implementation.
#
# We use:
# - token embeddings
# - learned position embeddings
# - a few decoder blocks
# - a language-model head
#
# The point is to have a realistic-enough model for training-infra demos.

def make_causal_mask(seq_len: int, device: torch.device) -> torch.Tensor:
    """Return a boolean lower-triangular causal mask."""
    return torch.tril(torch.ones(seq_len, seq_len, dtype=torch.bool, device=device))

class CausalSelfAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int, dropout: float = 0.1):
        super().__init__()

        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"

        self.d_model = d_model
        self.num_heads = num_heads
        self.head_dim = d_model // num_heads

        # Standard Q, K, V projections.
        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)

        # Final projection back to model dimension.
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout_p = dropout
        self.resid_dropout = nn.Dropout(dropout)

    def _split_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, seq_len, d_model]
        into     [batch, heads, seq_len, head_dim]
        """
        batch_size, seq_len, _ = x.shape
        x = x.view(batch_size, seq_len, self.num_heads, self.head_dim)
        x = x.transpose(1, 2)
        return x

    def _merge_heads(self, x: torch.Tensor) -> torch.Tensor:
        """
        Convert [batch, heads, seq_len, head_dim]
        back into [batch, seq_len, d_model]
        """
        batch_size, _, seq_len, _ = x.shape
        x = x.transpose(1, 2).contiguous()
        x = x.view(batch_size, seq_len, self.d_model)
        return x

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Project input into Q, K, V.
        q = self._split_heads(self.q_proj(x))
        k = self._split_heads(self.k_proj(x))
        v = self._split_heads(self.v_proj(x))

        # For training, we use a standard causal mask over the full sequence.
        seq_len = x.size(1)
        attn_mask = make_causal_mask(seq_len=seq_len, device=x.device)

        # PyTorch's scaled_dot_product_attention expects:
        # True  = allowed
        # False = blocked
        attn_out = F.scaled_dot_product_attention(
            q,
            k,
            v,
            attn_mask=attn_mask,
            dropout_p=self.dropout_p if self.training else 0.0,
            is_causal=False,  # we already passed the explicit mask
        )

        # Merge heads and project back.
        attn_out = self._merge_heads(attn_out)
        attn_out = self.resid_dropout(self.out_proj(attn_out))
        return attn_out

class FeedForward(nn.Module):
    def __init__(self, d_model: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.fc1 = nn.Linear(d_model, ff_dim)
        self.fc2 = nn.Linear(ff_dim, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # GELU is common in modern Transformer implementations.
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_dim: int, dropout: float = 0.1):
        super().__init__()

        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model=d_model, num_heads=num_heads, dropout=dropout)

        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model=d_model, ff_dim=ff_dim, dropout=dropout)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Pre-norm attention block.
        x = x + self.dropout(self.attn(self.ln1(x)))

        # Pre-norm feed-forward block.
        x = x + self.dropout(self.ffn(self.ln2(x)))
        return x

class TinyCausalLM(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        d_model: int = 32,
        num_heads: int = 4,
        ff_dim: int = 64,
        num_layers: int = 2,
        dropout: float = 0.1,
        max_len: int = 64,
    ):
        super().__init__()

        self.d_model = d_model
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_len, d_model)
        self.embed_dropout = nn.Dropout(dropout)

        self.layers = nn.ModuleList([
            DecoderBlock(d_model=d_model, num_heads=num_heads, ff_dim=ff_dim, dropout=dropout)
            for _ in range(num_layers)
        ])

        self.final_ln = nn.LayerNorm(d_model)

        # Tie output weights to token embeddings.
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_embed.weight

    def forward(self, input_ids: torch.Tensor) -> torch.Tensor:
        batch_size, seq_len = input_ids.shape

        # Build token + position embeddings.
        positions = torch.arange(seq_len, device=input_ids.device)
        x = self.token_embed(input_ids) * math.sqrt(self.d_model)
        x = x + self.pos_embed(positions)[None, :, :]
        x = self.embed_dropout(x)

        # Pass through decoder blocks.
        for layer in self.layers:
            x = layer(x)

        # Final normalization and LM head.
        x = self.final_ln(x)
        logits = self.lm_head(x)
        return logits

# Instantiate one small model and verify shapes.
model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.1,
    max_len=64,
).to(device)

input_ids, target_ids = make_counting_batch(batch_size=4, device=device)
logits = model(input_ids)

print("Input shape: ", tuple(input_ids.shape))
print("Target shape:", tuple(target_ids.shape))
print("Logits shape:", tuple(logits.shape))

assert logits.shape[:2] == input_ids.shape
assert logits.size(-1) == VOCAB_SIZE



## 2. Baseline single-device training

Before talking about distributed or sharded training, always start with the baseline:
- one process
- one device
- one model
- one optimizer
- one loss

If you cannot explain the single-device training loop clearly, the distributed version will feel much harder.

### Main ingredients
- forward pass
- loss computation
- backward pass
- optimizer step
- gradient clipping (optional but common)


In [ ]:

# ============================================================
# Baseline single-device training loop
# ============================================================

@dataclass
class TrainConfig:
    batch_size: int = 8
    learning_rate: float = 2e-3
    grad_clip_norm: float = 1.0
    cpu_steps: int = 6
    gpu_steps: int = 12

train_config = TrainConfig()

# Keep the demo short so the notebook stays responsive.
num_train_steps = train_config.gpu_steps if device.type == "cuda" else train_config.cpu_steps

optimizer = torch.optim.Adam(model.parameters(), lr=train_config.learning_rate)

# Ignore padding tokens because they are batching artifacts.
loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

def train_one_step(model: nn.Module) -> float:
    """Run one next-token training step and return the scalar loss."""
    model.train()

    input_ids, target_ids = make_counting_batch(
        batch_size=train_config.batch_size,
        device=device,
    )

    logits = model(input_ids)

    # CrossEntropyLoss expects:
    #   logits  -> [N, C]
    #   target  -> [N]
    # so we flatten batch and time together.
    loss = loss_fn(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
    )

    optimizer.zero_grad()
    loss.backward()

    # Gradient clipping is common in Transformer training.
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=train_config.grad_clip_norm)

    optimizer.step()
    return float(loss.item())

loss_history: List[float] = []

train_start = time.perf_counter()
for _ in range(num_train_steps):
    loss_history.append(train_one_step(model))
train_elapsed = time.perf_counter() - train_start

print(f"Ran {num_train_steps} steps on {device}.")
print(f"Elapsed time: {train_elapsed:.3f} sec")
print("Loss history:", [round(v, 4) for v in loss_history])

plt.figure(figsize=(6, 3))
plt.plot(loss_history, marker="o")
plt.title("Baseline training loss")
plt.xlabel("Step")
plt.ylabel("Cross-entropy loss")
plt.grid(True, alpha=0.3)
plt.show()


## Interview Topic 3a: Gradient Accumulation Mechanics

**Why this is critical for interviews:**
- ~80% of multi-GPU DDP/FSDP interviews ask about this
- Interview questions: "How do you train with effective batch size 2048 on 8 GPUs with 32GB memory?" Answer: gradient accumulation
- Learning rate scaling: `new_lr = old_lr × (new_eff_batch / old_eff_batch)^0.5` is a common test

### Core Idea
In distributed training, **effective batch size** = `local_batch_size × accumulation_steps × world_size`

Instead of:
- one large forward (runs out of memory)

You do:
- N smaller forwards + backwards
- accumulate gradients
- one optimizer step every N iterations

### Why It Matters
1. **Memory efficiency** - Use smaller micro-batch to fit in VRAM
2. **Global batch size control** - Hit target effective batch despite hardware constraints
3. **Learning rate correspondence** - Larger batches → need larger learning rate
4. **Convergence** - Wrong effective batch size → wrong convergence curve

### Interview Key Points
- "Gradient accumulation is only for memory, not compute efficiency on single GPU" (FALSE - it's about batch size)
- "You can't accumulate gradients for 4 steps and use batch_size=256, then expect it matches batch_size=1024 with one step" (TRUE - different gradient scaling)
- "When you accumulate N steps, you must divide loss by N if using reduction='sum'" (TRUE - need to normalize)

In [ ]:
# ============================================================
# Gradient Accumulation Demo
# ============================================================
# This demo shows that N accumulation steps of batch B
# is mathematically equivalent to one step of batch (N*B),
# IF you handle the loss scaling correctly.
#
# Key insight: accumulating gradients ≠ automatic equivalence.
# You must divide loss by accumulation_steps if using reduction='sum'.

torch.manual_seed(SEED)

# Build two models with identical weights
accum_model = build_deterministic_model()
large_batch_model = copy.deepcopy(accum_model)

# ===== Reference: one step with large batch =====
large_batch_size = 16
large_batch_input, large_batch_target = make_counting_batch(
    batch_size=large_batch_size,
    device=torch.device("cpu"),
)

large_logits = large_batch_model(large_batch_input)
large_loss = F.cross_entropy(
    large_logits.reshape(-1, VOCAB_SIZE),
    large_batch_target.reshape(-1),
    ignore_index=PAD_ID,
    reduction="mean",  # Convention: use mean for comparison
)
large_loss.backward()

large_batch_grads = {
    name: param.grad.detach().clone()
    for name, param in large_batch_model.named_parameters()
    if param.grad is not None
}

# ===== Accumulation: 4 steps of batch 4 =====
accumulation_steps = 4
micro_batch_size = 4
accum_optimizer = torch.optim.Adam(accum_model.parameters(), lr=1e-3)

accumulated_loss = 0.0
for accum_step in range(accumulation_steps):
    accum_optimizer.zero_grad()  # Standard pattern: zero before each forward

    micro_input, micro_target = make_counting_batch(
        batch_size=micro_batch_size,
        device=torch.device("cpu"),
    )

    micro_logits = accum_model(micro_input)
    # Key: if using reduction='sum', divide by accum_steps to get correct scale
    # Here we use reduction='mean' which PyTorch does per-batch, so behavior differs
    micro_loss = F.cross_entropy(
        micro_logits.reshape(-1, VOCAB_SIZE),
        micro_target.reshape(-1),
        ignore_index=PAD_ID,
        reduction="mean",
    )

    # Important: backward without step
    # This accumulates gradients into param.grad
    micro_loss.backward()
    accumulated_loss += micro_loss.item() / accumulation_steps

accum_batch_grads = {
    name: param.grad.detach().clone()
    for name, param in accum_model.named_parameters()
    if param.grad is not None
}

# Compare gradients
# Note: they may not be identical because large_batch and accum_samples are different
# But the pattern is correct - same scaling factor applied
print("=== Gradient Accumulation Pattern ===")
print(f"Large batch size:         {large_batch_size}")
print(f"Micro batch size:         {micro_batch_size}")
print(f"Accumulation steps:       {accumulation_steps}")
print(f"Effective batch equiv:    {micro_batch_size * accumulation_steps}")
print(f"Large batch loss:         {large_loss.item():.6f}")
print(f"Accumulated avg loss:     {accumulated_loss:.6f}")


# ===== Effective batch size calculation demo =====
print("\n=== Effective Batch Size Calculation ===")

def calculate_effective_batch_and_lr_scaling(
    local_batch: int,
    accum_steps: int,
    world_size: int,
    reference_batch: int = 32,
    reference_lr: float = 0.001,
) -> Dict[str, float]:
    """
    Interview standard: Calculate effective batch and proper LR scaling.
    
    Rule of thumb (for large batches, use sqrt scaling):
    new_lr = old_lr * sqrt(new_eff_batch / old_eff_batch)
    """
    eff_batch = local_batch * accum_steps * world_size
    lr_scale_factor = math.sqrt(eff_batch / reference_batch)
    new_lr = reference_lr * lr_scale_factor

    return {
        "effective_batch_size": eff_batch,
        "lr_scale_factor": lr_scale_factor,
        "new_learning_rate": new_lr,
    }

# Example scenarios common in interviews
scenarios = [
    {"local_batch": 32, "accum_steps": 1, "world_size": 1, "name": "Single GPU baseline"},
    {"local_batch": 32, "accum_steps": 1, "world_size": 8, "name": "DDP 8 GPUs, no accum"},
    {"local_batch": 8, "accum_steps": 4, "world_size": 8, "name": "DDP 8 GPUs + accum 4x"},
    {"local_batch": 4, "accum_steps": 8, "world_size": 16, "name": "DDP 16 GPUs + accum 8x"},
]

for scenario in scenarios:
    result = calculate_effective_batch_and_lr_scaling(
        local_batch=scenario["local_batch"],
        accum_steps=scenario["accum_steps"],
        world_size=scenario["world_size"],
    )
    print(f"\n{scenario['name']}:")
    print(f"  Effective batch: {result['effective_batch_size']}")
    print(f"  LR scale factor: {result['lr_scale_factor']:.3f}x")
    print(f"  New LR: {result['new_learning_rate']:.2e}")

### Exercise 3a.1 — Gradient Accumulation Convergence

You have 8 GPUs. Two training configs:
- **Config A:** local_batch=64, accum_steps=1, LR=0.001
- **Config B:** local_batch=16, accum_steps=4, LR=0.001

Both have effective batch = 512 (before LR scaling).

Should Config B use the same LR as Config A? Why or why not?

### Exercise 3a.1 Answer

**NO.** Config B should scale its learning rate.

**Why:**
- Both configs have the same effective batch (512), but they get there differently
- Same effective batch ≠ identical gradient statistics
- Config A: 512 samples in one synchronization
- Config B: 128 samples × 4 → 4 gradient updates → different convergence dynamics
- Standard practice: `new_lr = old_lr × sqrt(new_eff_batch / old_eff_batch)`
- If old reference was batch=256, both should scale: `sqrt(512/256) ≈ 1.41`

**Key insight:** Accumulation steps change the number of synchronized updates per epoch, which affects the effective learning rate schedule.


## 3. DDP intuition

**DistributedDataParallel (DDP)** is the standard starting point for multi-GPU training.

### Core idea
- each rank owns a **full model replica**
- each rank processes a **different data shard**
- after backward, gradients are synchronized across ranks
- then all ranks take the same optimizer step

### Mental model
DDP is basically:
- "multiple model replicas"
- "different mini-batches per rank"
- "gradient averaging / all-reduce"

### Very important interview point
DDP does **not** automatically shard the input for you.
You typically use a **DistributedSampler** so each rank gets a different piece of data.


## Interview Topic 3b: Global Gradient Norm Clipping

**Critical for interviews:**
- ~65% of stability-focused interviews ask about this
- Common failure: "My DDP training diverges while single-GPU works" → often missing global clipping
- Test question: "You clip gradients locally on each rank. Is that equivalent to clipping globally?" Answer: NO

### The Problem
- **Local clipping:** Each rank clips its own gradients independently
- **Global clipping:** Compute gradient norm across ALL ranks, then clip globally

At large scale with many ranks:
- Different ranks may have different gradient magnitudes
- Local clipping can allow very large gradients to slip through on one rank
- Global clipping ensures stable training across all ranks

### Why DDP Needs Global Clipping
In DDP, after backward:
- Rank 0 has gradients g0
- Rank 1 has gradients g1
- ...

If you do local clipping:
```
rank 0: clip(g0) to norm=1.0
rank 1: clip(g1) to norm=1.0
```

Then AllReduce gradients:
```
all_reduce([clip(g0), clip(g1)]) 
```

**Problem:** The clipped gradients from rank 1 could still be huge if g1's norm was large!

**Solution:** Compute norm across all ranks, then clip:
```
global_norm = sqrt(sum(||g_i||^2 for all ranks))
if global_norm > clip_threshold:
    for all ranks: g_i *= clip_threshold / global_norm
```

In [ ]:
# ============================================================
# Global Gradient Norm Clipping Demo
# ============================================================
# This demo shows the difference between local and global clipping
# by simulating two "ranks" with different gradient magnitudes.

def compute_grad_norm(model: nn.Module) -> float:
    """Compute L2 norm of all gradients in model."""
    total_norm_sq = 0.0
    for param in model.parameters():
        if param.grad is not None:
            total_norm_sq += (param.grad.data ** 2).sum().item()
    return math.sqrt(total_norm_sq)

def local_clip_grad_norm(model: nn.Module, max_norm: float) -> float:
    """Local clipping: each rank clips independently."""
    norm = compute_grad_norm(model)
    if norm > max_norm:
        scale = max_norm / (norm + 1e-8)
        for param in model.parameters():
            if param.grad is not None:
                param.grad.data *= scale
    return norm

def global_clip_grad_norm_simulation(
    models: list,
    max_norm: float,
    world_size: int,
) -> list:
    """
    Simulate global clipping across multiple 'ranks'.
    
    In real distributed training, you'd use dist.all_reduce on grad norm.
    Here we simulate by collecting all norms and computing globally.
    """
    # Step 1: Compute local norms
    local_norms = [compute_grad_norm(m) for m in models]

    # Step 2: Compute global norm (AllReduce simulation)
    # global_norm = sqrt(sum(local_norm_i^2))
    global_norm_sq = sum(n**2 for n in local_norms)
    global_norm = math.sqrt(global_norm_sq)

    # Step 3: Clip if needed
    if global_norm > max_norm:
        scale = max_norm / (global_norm + 1e-8)
        for model in models:
            for param in model.parameters():
                if param.grad is not None:
                    param.grad.data *= scale

    return local_norms, global_norm

# ===== Setup: Two models with intentionally different gradient magnitudes =====
torch.manual_seed(SEED)

# Rank 0: Small gradients
rank0_model = build_deterministic_model()
rank0_input, rank0_target = make_counting_batch(batch_size=4, device=torch.device("cpu"))
rank0_logits = rank0_model(rank0_input)
rank0_loss = F.cross_entropy(
    rank0_logits.reshape(-1, VOCAB_SIZE),
    rank0_target.reshape(-1),
    ignore_index=PAD_ID,
    reduction="mean",
)
rank0_loss.backward()

# Rank 1: Large gradients (scale up by 10x to simulate different batch dynamics)
rank1_model = copy.deepcopy(rank0_model)
rank1_input, rank1_target = make_counting_batch(batch_size=4, device=torch.device("cpu"))
rank1_logits = rank1_model(rank1_input)
rank1_loss = F.cross_entropy(
    rank1_logits.reshape(-1, VOCAB_SIZE),
    rank1_target.reshape(-1),
    ignore_index=PAD_ID,
    reduction="mean",
)
rank1_loss.backward()

# Scale rank1 gradients to simulate different magnitude
for param in rank1_model.parameters():
    if param.grad is not None:
        param.grad.data *= 10.0

before_clip_norm0 = compute_grad_norm(rank0_model)
before_clip_norm1 = compute_grad_norm(rank1_model)

print("=== Before Clipping ===")
print(f"Rank 0 grad norm: {before_clip_norm0:.4f}")
print(f"Rank 1 grad norm: {before_clip_norm1:.4f}")
print(f"Global grad norm: {math.sqrt(before_clip_norm0**2 + before_clip_norm1**2):.4f}")

# ===== Strategy A: Local clipping (WRONG at scale) =====
rank0_model_local = copy.deepcopy(rank0_model)
rank1_model_local = copy.deepcopy(rank1_model)

local_clip_grad_norm(rank0_model_local, max_norm=1.0)
local_clip_grad_norm(rank1_model_local, max_norm=1.0)

print("\n=== After Local Clipping (per-rank) ===")
print(f"Rank 0 grad norm: {compute_grad_norm(rank0_model_local):.4f}")
print(f"Rank 1 grad norm: {compute_grad_norm(rank1_model_local):.4f}")
print(f"Combined (if AllReduced): {math.sqrt(compute_grad_norm(rank0_model_local)**2 + compute_grad_norm(rank1_model_local)**2):.4f}")

# ===== Strategy B: Global clipping (CORRECT) =====
rank0_model_global = copy.deepcopy(rank0_model)
rank1_model_global = copy.deepcopy(rank1_model)

local_norms, global_norm = global_clip_grad_norm_simulation(
    [rank0_model_global, rank1_model_global],
    max_norm=1.0,
    world_size=2,
)

print("\n=== After Global Clipping ===")
print(f"Rank 0 grad norm: {compute_grad_norm(rank0_model_global):.4f}")
print(f"Rank 1 grad norm: {compute_grad_norm(rank1_model_global):.4f}")
print(f"Combined: {math.sqrt(compute_grad_norm(rank0_model_global)**2 + compute_grad_norm(rank1_model_global)**2):.4f}")

print("\n=== Comparison ===")
print(f"Local clipping keeps global norm at:  {math.sqrt(compute_grad_norm(rank0_model_local)**2 + compute_grad_norm(rank1_model_local)**2):.4f}")
print(f"Global clipping keeps global norm at: {math.sqrt(compute_grad_norm(rank0_model_global)**2 + compute_grad_norm(rank1_model_global)**2):.4f}")
print("\nConclusion: Global clipping respects the GLOBAL threshold across all ranks!")

### Exercise 3b.1 — When Does Local vs Global Clipping Matter?

You have 1000 GPUs training with DDP.

**Scenario:** One GPU has a gradient norm of 100 (perhaps its batch had some outliers), and 999 GPUs have gradient norm 1.0.

With local clipping to max_norm=1.0:
- 999 GPUs: norm stays ~1.0
- 1 GPU: norm gets clipped to 1.0

After AllReduce of gradients, is the global gradient norm close to 1.0?

Why or why not?

### Exercise 3b.1 Answer

**NO.** The global gradient norm will be much larger than 1.0.

**Why:**
- After AllReduce, you're summing 1000 gradients
- Each of the 999 normal ranks contributes ~1.0 norm gradients
- The problematic rank also contributes ~1.0 (after local clipping)
- Global norm = sqrt(1.0^2 + 1.0^2 + ... + 1.0^2) = sqrt(1000) ≈ 31.6

**So you end up with global norm ≈ 31.6 when you wanted ≤ 1.0!**

This is a common silent failure in large-scale training:
- Locally clipped gradients look stable
- But globally, they're unstable
- Model training becomes jittery or diverges

**Fix:** Use global clipping with AllReduce:
```
global_norm = sqrt(AllReduce(local_norm^2))
if global_norm > max_norm:
    scale = max_norm / global_norm
    for all ranks: scale all gradients by scale
```

In [ ]:

# ============================================================
# DDP intuition demo: gradient averaging == larger batch
# ============================================================
# This is one of the most useful DDP interview demos you can know.
#
# We will:
# 1) create one model and compute gradients on a full batch
# 2) clone the same model twice
# 3) split the batch across two "fake ranks"
# 4) compute gradients on each half independently
# 5) combine those gradients carefully
# 6) show they match the full-batch gradients
#
# Important nuance:
# If different ranks see different numbers of valid tokens
# (because of variable sequence lengths and padding),
# you must be careful about reduction conventions.
#
# To keep the comparison clean and mathematically exact, we:
# - use cross-entropy with reduction="sum"
# - divide by the global number of non-pad targets

def build_deterministic_model() -> TinyCausalLM:
    # Use dropout=0 so the gradients are fully deterministic for comparison.
    return TinyCausalLM(
        vocab_size=VOCAB_SIZE,
        d_model=32,
        num_heads=4,
        ff_dim=64,
        num_layers=2,
        dropout=0.0,
        max_len=64,
    )

torch.manual_seed(SEED)
base_model = build_deterministic_model()

# Build one batch that we will later split into two equal halves.
# Variable sequence lengths are okay because we are going to handle
# token-count normalization explicitly.
full_input_ids, full_target_ids = make_counting_batch(batch_size=8, device=torch.device("cpu"))

def count_non_pad_tokens(target_ids: torch.Tensor) -> int:
    return int((target_ids != PAD_ID).sum().item())

global_non_pad = count_non_pad_tokens(full_target_ids)

# ----- Reference gradients from a single larger batch -----
single_model = copy.deepcopy(base_model)
single_logits = single_model(full_input_ids)

single_loss_sum = F.cross_entropy(
    single_logits.reshape(-1, VOCAB_SIZE),
    full_target_ids.reshape(-1),
    ignore_index=PAD_ID,
    reduction="sum",
)

single_loss = single_loss_sum / global_non_pad
single_loss.backward()

single_grads = {
    name: param.grad.detach().clone()
    for name, param in single_model.named_parameters()
    if param.grad is not None
}

# ----- Simulate two DDP ranks -----
rank0_model = copy.deepcopy(base_model)
rank1_model = copy.deepcopy(base_model)

rank0_input, rank1_input = full_input_ids.chunk(2, dim=0)
rank0_target, rank1_target = full_target_ids.chunk(2, dim=0)

rank0_non_pad = count_non_pad_tokens(rank0_target)
rank1_non_pad = count_non_pad_tokens(rank1_target)

rank0_logits = rank0_model(rank0_input)
rank0_loss_sum = F.cross_entropy(
    rank0_logits.reshape(-1, VOCAB_SIZE),
    rank0_target.reshape(-1),
    ignore_index=PAD_ID,
    reduction="sum",
)

rank1_logits = rank1_model(rank1_input)
rank1_loss_sum = F.cross_entropy(
    rank1_logits.reshape(-1, VOCAB_SIZE),
    rank1_target.reshape(-1),
    ignore_index=PAD_ID,
    reduction="sum",
)

# Scale each local loss by the same global denominator so that
# summing / averaging rank gradients reproduces the global gradient.
(rank0_loss_sum / global_non_pad).backward()
(rank1_loss_sum / global_non_pad).backward()

# Sum the per-rank gradients. Because each rank used the same
# global denominator, the summed gradients should match the
# full-batch gradients.
combined_grads = {}
for (name0, p0), (name1, p1) in zip(rank0_model.named_parameters(), rank1_model.named_parameters()):
    assert name0 == name1
    if p0.grad is None:
        continue
    combined_grads[name0] = p0.grad.detach() + p1.grad.detach()

max_abs_diff = 0.0
for name in single_grads:
    diff = (single_grads[name] - combined_grads[name]).abs().max().item()
    max_abs_diff = max(max_abs_diff, diff)

print(f"Global non-pad token count: {global_non_pad}")
print(f"Rank0 non-pad tokens:       {rank0_non_pad}")
print(f"Rank1 non-pad tokens:       {rank1_non_pad}")
print(f"Single-batch normalized loss: {(single_loss_sum / global_non_pad).item():.6f}")
print(f"Max abs grad diff:            {max_abs_diff:.8f}")
print("DDP intuition check passed:", max_abs_diff < 1e-5)


In [ ]:

# ============================================================
# DistributedSampler intuition demo
# ============================================================
# DDP does not automatically decide which samples each rank gets.
# A DistributedSampler-like partitioning is what avoids duplicate work.

def simple_distributed_partition(dataset_size: int, world_size: int, rank: int) -> List[int]:
    """Simple round-robin partitioning to illustrate the idea."""
    return list(range(rank, dataset_size, world_size))

dataset_size = 12
world_size = 3

for rank in range(world_size):
    shard = simple_distributed_partition(dataset_size, world_size, rank)
    print(f"Rank {rank} sees indices: {shard}")


## Interview Topic 3c: Multi-Node Training Considerations

**Why this critical for senior interviews:**
- Most real training happens on 10-1000 nodes
- ~60% of infrastructure interviews include this
- Common gap: L5 candidates assume multi-node is just scaled DDP. It's not.

### Key Differences: Single-Node vs Multi-Node

| Aspect | Single-Node | Multi-Node |
|--------|-------------|-----------|
| **Communication** | NVLink/PCIe (10-100 GB/s) | Ethernet (10-100 Gb/s, often with congestion) |
| **Latency** | Microseconds | Milliseconds (100-1000x worse) |
| **Synchronization cost** | Cheap (one machine's clock) | Expensive (network RTT) |
| **Failure modes** | Device crash | Network partition, rank hang, timeout |
| **NUMA effects** | Possible on socket | Cross-socket memory access 3-5x slower |
| **Typical bottleneck** | Compute | Communication during gradient AllReduce |

### Interview Question

**Interviewer asks:** "You have 8 GPU training code that trains perfectly on one 8-GPU node in 2 hours per epoch. You scale it to 128 GPUs on 16 nodes. You expect ~4 min per epoch (8x speedup). Instead it takes 20 minutes per epoch. What went wrong?"

**Wrong answers:**
- "We need more memory" (memory scales fine)
- "The model is too small" (bigger model would help but doesn't solve it)

**Correct analysis:**
- Single node: 7 gradient AllReduce operations (one AllReduce per backward)  
- Each AllReduce is ~10 GB across NVLink → ~50 microseconds
- 16-node cluster: AllReduce is ~10 GB over Ethernet → ~1-10 milliseconds (100-200x worse)
- If you do AllReduce after every batch: 1000 batches × 10ms = 10 seconds overhead
- Plus 100+ batches: significant fraction of epoch time

**Solution approaches:**
1. **Gradient accumulation** - Do 4 AllReduces instead of 1000
2. **Communication overlap** - Start AllReduce while still computing backward
3. **Sparse synchronization** - Only AllReduce every N steps
4. **Better interconnect** - InfiniBand instead of Ethernet

### NUMA Effects (Often Overlooked)

On a single GPU node with 2 sockets:
- GPU on socket 0
- Your main CPU rank process on socket 1
- Memory allocation: socket 1 is closer
- When GPU needs data from socket 0 RAM: 3-5x slower

Interview insight: "My 8-GPU node training is not CPU-bound, but peak throughput is 40% lower than expected" → Often NUMA pinning issue.

In [ ]:
# ============================================================
# Multi-Node Communication Cost Analysis
# ============================================================
# This is a rough calculation to show why multi-node is harder.

def estimate_allreduce_time_ms(
    data_size_gb: float,
    interconnect: str,
) -> float:
    """
    Rough estimate of AllReduce time based on interconnect.
    Real measurements vary widely, but this shows the scale.
    """
    interconnects = {
        "nvlink": {
            "bandwidth_gbps": 900,  # GB/s
            "latency_us": 1,
        },
        "pcie": {
            "bandwidth_gbps": 16,
            "latency_us": 5,
        },
        "ethernet_100gb": {
            "bandwidth_gbps": 10,
            "latency_us": 500,
        },
        "ethernet_10gb": {
            "bandwidth_gbps": 1,
            "latency_us": 1000,
        },
        "infiniband": {
            "bandwidth_gbps": 200,
            "latency_us": 100,
        },
    }

    if interconnect not in interconnects:
        raise ValueError(f"Unknown interconnect: {interconnect}")

    spec = interconnects[interconnect]
    bandwidth_gbps = spec["bandwidth_gbps"]
    latency_us = spec["latency_us"]

    # Simple model: AllReduce time ≈ latency + data_size / bandwidth
    # In reality, tree algorithms and ring algorithms have different characteristics
    bandwidth_us = (data_size_gb * 1000) / bandwidth_gbps  # Convert to microseconds
    total_us = latency_us + bandwidth_us

    return total_us / 1000.0  # Convert to milliseconds

# Scenario: Baseline 8-GPU training, scaled to 16 nodes
print("=== AllReduce Cost Analysis ===\n")

# Single 8-GPU node
print("Single 8-GPU node (NVLink):")
time_nvlink = estimate_allreduce_time_ms(10.0, "nvlink")
print(f"  10 GB AllReduce: {time_nvlink:.3f} ms = {time_nvlink*1000:.1f} µs\n")

# Multi-node options
print("Multi-node (16 nodes × 8 GPUs = 128 GPUs):")
for interconnect in ["ethernet_10gb", "ethernet_100gb", "infiniband"]:
    time_ms = estimate_allreduce_time_ms(10.0, interconnect)
    print(f"  {interconnect:20s}: {time_ms:.3f} ms (= {time_ms/time_nvlink:.0f}x slower)")

# Impact on epoch time
print("\n=== Epoch Time Impact ===\n")

def epoch_time_estimate(
    num_batches: int,
    batch_forward_backward_ms: float,
    allreduce_cost_ms: float,
    num_allreduces_per_batch: int,
) -> dict:
    """Estimate total epoch time with communication overhead."""
    compute_time = num_batches * batch_forward_backward_ms
    comm_time = num_batches * num_allreduces_per_batch * allreduce_cost_ms
    total_time = compute_time + comm_time

    return {
        "compute_time": compute_time,
        "comm_time": comm_time,
        "total_time": total_time,
        "comm_fraction": comm_time / total_time,
    }

# Realistic scenario for baseline model
num_batches = 1000
batch_time_ms = 100  # 100ms per forward+backward+optimizer on one GPU
num_allreduces = 1  # One AllReduce per gradient step (synchronized)

print("Baseline: 1000 batches, 100ms per batch forward/backward\n")

result_single = epoch_time_estimate(
    num_batches=num_batches,
    batch_forward_backward_ms=batch_time_ms,
    allreduce_cost_ms=0.001,  # NVLink: ~1 microsecond
    num_allreduces_per_batch=num_allreduces,
)

result_multi_10gb = epoch_time_estimate(
    num_batches=num_batches,
    batch_forward_backward_ms=batch_time_ms / 8,  # 8 GPUs on single node → 125ms becomes 100/8 + overhead
    allreduce_cost_ms=10,  # 10 Gb Ethernet: ~10ms
    num_allreduces_per_batch=num_allreduces,
)

result_multi_100gb = epoch_time_estimate(
    num_batches=num_batches,
    batch_forward_backward_ms=batch_time_ms / 8,
    allreduce_cost_ms=1,  # 100 Gb Ethernet: ~1ms
    num_allreduces_per_batch=num_allreduces,
)

result_infiniband = epoch_time_estimate(
    num_batches=num_batches,
    batch_forward_backward_ms=batch_time_ms / 8,
    allreduce_cost_ms=0.5,  # InfiniBand: ~0.5ms
    num_allreduces_per_batch=num_allreduces,
)

print(f"Single 8-GPU node (NVLink):")
print(f"  Compute time: {result_single['compute_time']/1000/60:.1f} min")
print(f"  Comm time:    {result_single['comm_time']/1000/60:.1f} min")
print(f"  Total:        {result_single['total_time']/1000/60:.1f} min")
print(f"  Comm fraction: {result_single['comm_fraction']:.1%}\n")

print(f"16-node (10 Gb Ethernet):")
print(f"  Compute time: {result_multi_10gb['compute_time']/1000/60:.1f} min")
print(f"  Comm time:    {result_multi_10gb['comm_time']/1000/60:.1f} min")
print(f"  Total:        {result_multi_10gb['total_time']/1000/60:.1f} min")
print(f"  Comm fraction: {result_multi_10gb['comm_fraction']:.1%}  ← BOTTLENECK!\n")

print(f"16-node (100 Gb Ethernet):")
print(f"  Compute time: {result_multi_100gb['compute_time']/1000/60:.1f} min")
print(f"  Comm time:    {result_multi_100gb['comm_time']/1000/60:.1f} min")
print(f"  Total:        {result_multi_100gb['total_time']/1000/60:.1f} min")
print(f"  Comm fraction: {result_multi_100gb['comm_fraction']:.1%}\n")

print(f"16-node (InfiniBand):")
print(f"  Compute time: {result_infiniband['compute_time']/1000/60:.1f} min")
print(f"  Comm time:    {result_infiniband['comm_time']/1000/60:.1f} min")
print(f"  Total:        {result_infiniband['total_time']/1000/60:.1f} min")
print(f"  Comm fraction: {result_infiniband['comm_fraction']:.1%}")

print("\n=== Key Takeaway ===")
print("With slow interconnect, communication becomes the dominant bottleneck.")
print("Single-node: ~99.9% compute")
print("16-node 10Gb: ~50% communication!")
print("This is why large-scale training infrastructure matters.")

### Exercise 3c.1 — Communication Bottleneck at Scale

You're training a 70B parameter LLM on 256 GPUs.
- Single GPU batch: 2 samples, takes 200ms (forward + backward)
- On 256 GPUs with 10 Gb Ethernet: AllReduce takes ~100ms

How many GPUs do you actually need before communication becomes the bottleneck (>30% of epoch time)?

**Hint:** As you add more ranks, AllReduce communication grows, but per-rank compute decreases.

### Exercise 3c.1 Answer

With 10 Gb Ethernet AllReduce ≈100ms:

| # GPUs | Per-GPU compute | AllReduce time | Comm fraction |
|-------|-----------------|----------------|---------------|
| 8 (1 node) | 200ms | ~0.01ms | <1% |
| 16 | 100ms | ~100ms | 50% |
| 32 | 50ms | ~100ms | 67% |
| 256 | 6.25ms | ~100ms | 94% |

**Answer: ~16 GPUs**, communication hits 50%.

**Why:** AllReduce cost doesn't scale linearly with number of GPUs.
- With tree all-reduce: communication ~ O(log N) rounds, not O(N)
- But tree have higher bandwidth requirements → often flat cost across 8-256 GPUs
- Per-rank compute drops as N increases (same total work / N ranks)
- So communication fraction ∝ 1/per_rank_compute ∝ grows as N increases

**Interview insight:** This is why gradient accumulation is essential at scale.
- Instead of 1000 AllReduces per epoch (1000 × 100ms = 100s)
- With 4x accumulation: 250 AllReduces (250 × 100ms = 25s overhead only)


### Optional real DDP launch skeleton

The next cell is a **reference snippet** for real multi-process DDP.  
It is included for interview familiarity, but it is **not executed** by default inside this notebook.

Why not execute it here?
- Colab usually gives one process / one notebook context
- true DDP normally uses `torchrun` with multiple processes


In [ ]:

# ============================================================
# Optional reference: real DDP skeleton (not executed by default)
# ============================================================
RUN_REAL_DDP = False

if RUN_REAL_DDP:
    import os
    import torch.distributed as dist
    from torch.nn.parallel import DistributedDataParallel as DDP
    from torch.utils.data import DataLoader, DistributedSampler, TensorDataset

    # In a real DDP run, torchrun sets environment variables such as:
    # - RANK
    # - WORLD_SIZE
    # - LOCAL_RANK
    rank = int(os.environ["RANK"])
    world_size = int(os.environ["WORLD_SIZE"])
    local_rank = int(os.environ["LOCAL_RANK"])

    # NCCL is typical for multi-GPU CUDA training.
    # Gloo is common for CPU demos.
    backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend=backend)

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        ddp_device = torch.device(f"cuda:{local_rank}")
    else:
        ddp_device = torch.device("cpu")

    ddp_model = TinyCausalLM(vocab_size=VOCAB_SIZE).to(ddp_device)
    ddp_model = DDP(
        ddp_model,
        device_ids=[local_rank] if ddp_device.type == "cuda" else None,
    )

    # Every rank should see a different shard of the data.
    dummy_x, dummy_y = make_counting_batch(batch_size=32, device=torch.device("cpu"))
    dataset = TensorDataset(dummy_x.cpu(), dummy_y.cpu())
    sampler = DistributedSampler(dataset, num_replicas=world_size, rank=rank, shuffle=True)
    loader = DataLoader(dataset, batch_size=8, sampler=sampler)

    optimizer = torch.optim.Adam(ddp_model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    ddp_model.train()
    for batch_input_ids, batch_target_ids in loader:
        batch_input_ids = batch_input_ids.to(ddp_device)
        batch_target_ids = batch_target_ids.to(ddp_device)

        logits = ddp_model(batch_input_ids)
        loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), batch_target_ids.reshape(-1))

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    dist.destroy_process_group()



## 4. FSDP intuition

**Fully Sharded Data Parallel (FSDP)** goes further than DDP.

### DDP
- every rank holds a full model replica
- gradients are synchronized across replicas

### FSDP
- parameters, gradients, and optimizer state are sharded across ranks
- each rank holds only a shard most of the time
- full parameters are gathered only when needed for compute

### Why this matters
At large scale, model memory becomes the main bottleneck.
FSDP reduces per-rank memory so you can train models that would not fit under pure DDP.


In [ ]:

# ============================================================
# FSDP intuition demo: rough memory accounting
# ============================================================
# This is a rough mental model, not an exact byte-for-byte runtime trace.
#
# The goal is to compare:
# - DDP: each rank keeps full params + full grads + full optimizer state
# - FSDP: these are sharded across ranks outside active computation

def count_trainable_parameters(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def bytes_to_mb(num_bytes: float) -> float:
    return num_bytes / (1024 ** 2)

# Build a somewhat larger toy model so the chart is more interesting.
memory_demo_model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=128,
    num_heads=8,
    ff_dim=256,
    num_layers=6,
    dropout=0.0,
    max_len=128,
)

num_params = count_trainable_parameters(memory_demo_model)

# Rough assumptions for interview intuition:
# - parameters stored in bf16/fp16 -> 2 bytes
# - gradients often kept in fp32 -> 4 bytes
# - Adam optimizer keeps two fp32 state tensors -> 8 bytes total
param_bytes_per_elem = 2
grad_bytes_per_elem = 4
adam_state_bytes_per_elem = 8

total_param_bytes = num_params * param_bytes_per_elem
total_grad_bytes = num_params * grad_bytes_per_elem
total_optim_bytes = num_params * adam_state_bytes_per_elem

def estimate_ddp_bytes_per_rank() -> float:
    # Each rank owns a full copy of params, grads, and optimizer state.
    return total_param_bytes + total_grad_bytes + total_optim_bytes

def estimate_fsdp_bytes_per_rank(world_size: int) -> float:
    # Rough idealized sharded-at-rest estimate.
    # Real FSDP has communication, temporary all-gathers, fragmentation,
    # and implementation details beyond this simple picture.
    return (total_param_bytes + total_grad_bytes + total_optim_bytes) / world_size

world_sizes = [1, 2, 4, 8]
ddp_mb = [bytes_to_mb(estimate_ddp_bytes_per_rank()) for _ in world_sizes]
fsdp_mb = [bytes_to_mb(estimate_fsdp_bytes_per_rank(ws)) for ws in world_sizes]

print(f"Trainable parameters: {num_params:,}")
print(f"Rough DDP per-rank memory:  {ddp_mb[0]:.2f} MB")

plt.figure(figsize=(7, 4))
plt.plot(world_sizes, ddp_mb, marker="o", label="DDP per rank (rough)")
plt.plot(world_sizes, fsdp_mb, marker="o", label="FSDP per rank (rough ideal)")
plt.title("Rough per-rank memory intuition: DDP vs FSDP")
plt.xlabel("World size")
plt.ylabel("Memory per rank (MB)")
plt.xticks(world_sizes)
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()


In [ ]:

# ============================================================
# FSDP lifecycle intuition
# ============================================================
# This is a textual simulation of the idea:
#
# At rest:
#   each rank stores only shards
#
# During compute:
#   FSDP may all-gather full parameters for a wrapped unit,
#   run compute,
#   then reshard again.

def print_fsdp_lifecycle() -> None:
    steps = [
        "1) At rest: each rank keeps only its shard of parameters.",
        "2) Before a wrapped unit runs forward: all-gather full parameters for that unit.",
        "3) Run forward computation on the full parameters.",
        "4) Free or reshard full parameters after that unit finishes.",
        "5) During backward: gather what is needed, compute gradients.",
        "6) Reduce / shard gradients and keep sharded optimizer state.",
    ]
    for step in steps:
        print(step)

print_fsdp_lifecycle()



### Optional real FSDP launch skeleton

The next cell is a **reference snippet** for real FSDP.  
It is not executed automatically in this notebook.

One important interview detail:
- with FSDP, you usually **wrap first**
- then create the optimizer **after wrapping**
because the wrapped module changes parameter handling


In [ ]:

# ============================================================
# Optional reference: real FSDP skeleton (not executed by default)
# ============================================================
RUN_REAL_FSDP = False

if RUN_REAL_FSDP:
    import os
    import torch.distributed as dist
    from torch.distributed.fsdp import FullyShardedDataParallel as FSDP

    rank = int(os.environ["RANK"])
    local_rank = int(os.environ["LOCAL_RANK"])

    backend = "nccl" if torch.cuda.is_available() else "gloo"
    dist.init_process_group(backend=backend)

    if torch.cuda.is_available():
        torch.cuda.set_device(local_rank)
        fsdp_device = torch.device(f"cuda:{local_rank}")
    else:
        fsdp_device = torch.device("cpu")

    fsdp_model = TinyCausalLM(vocab_size=VOCAB_SIZE).to(fsdp_device)

    # Important: wrap the model before constructing the optimizer.
    fsdp_model = FSDP(fsdp_model, device_id=fsdp_device if fsdp_device.type == "cuda" else None)

    optimizer = torch.optim.Adam(fsdp_model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

    input_ids, target_ids = make_counting_batch(batch_size=8, device=fsdp_device)

    fsdp_model.train()
    logits = fsdp_model(input_ids)
    loss = loss_fn(logits.reshape(-1, VOCAB_SIZE), target_ids.reshape(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    dist.destroy_process_group()



## 5. AMP intuition

**Automatic Mixed Precision (AMP)** is about using lower precision where it helps performance, while still keeping enough numerical stability.

### Main ideas
- use `autocast` to let PyTorch pick lower precision for eligible ops
- on CUDA fp16 training, use `GradScaler` to reduce underflow risk
- on CPU or bf16 flows, you may use autocast without scaling

### Why infra engineers care
AMP affects:
- speed
- memory footprint
- numerical behavior
- hardware utilization


In [ ]:

# ============================================================
# AMP demo
# ============================================================
# This cell shows:
# - how to enter an autocast region
# - how output dtypes can change
# - how a training step looks with or without GradScaler
#
# Note:
# - On CUDA we usually use float16 or bfloat16 autocast.
# - On CPU, bfloat16 autocast is the common teaching/demo path.

amp_model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.0,
    max_len=64,
).to(device)

amp_optimizer = torch.optim.Adam(amp_model.parameters(), lr=1e-3)
amp_loss_fn = nn.CrossEntropyLoss(ignore_index=PAD_ID)

amp_input_ids, amp_target_ids = make_counting_batch(batch_size=4, device=device)

# Choose an autocast dtype based on device.
if device.type == "cuda":
    autocast_dtype = torch.float16
else:
    autocast_dtype = torch.bfloat16

# GradScaler is mainly useful for float16 CUDA training.
use_grad_scaler = device.type == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_grad_scaler else None

amp_model.train()
amp_optimizer.zero_grad()

with torch.autocast(device_type=device.type, dtype=autocast_dtype):
    amp_logits = amp_model(amp_input_ids)
    amp_loss = amp_loss_fn(
        amp_logits.reshape(-1, VOCAB_SIZE),
        amp_target_ids.reshape(-1),
    )

if scaler is not None:
    # Scaled backward and optimizer step.
    scaler.scale(amp_loss).backward()
    scaler.step(amp_optimizer)
    scaler.update()
else:
    # CPU / bf16 path: autocast only, standard backward.
    amp_loss.backward()
    amp_optimizer.step()

print(f"Autocast dtype requested: {autocast_dtype}")
print(f"Logits dtype under autocast: {amp_logits.dtype}")
print(f"AMP loss is finite: {bool(torch.isfinite(amp_loss).item())}")


## Interview Topic 4: Loss Scaling Deep Dive

**Critical for all AMP/fp16 interviews (~70% ask about this):**
- "Why does your loss turn to NaN after 10K steps?" → usually loss scaling issue
- "How do you detect overflow in fp16 training?" → GradScaler behavior
- "Why does loss scaling decay?" → gradient overflow detection

### The Fundamental Problem

fp16 range is **much narrower** than fp32:
- fp32: exponent can represent ~1e-38 to 1e38
- fp16: exponent can represent ~1e-4 to 1e4

In backprop with fp16:
- Loss is small (e.g., 2.5)
- Gradients are often smaller than loss
- When computing gradients in fp16: values < 6.1e-5 underflow to zero
- Result: **no gradient updates** (silent gradient death)

### Solution: Scale Loss Before Backward

```
scaled_loss = loss × scale_factor   (e.g., scale_factor = 2^15 = 32768)
scaled_loss.backward()              (gradients now 32768x larger)
gradients /= scale_factor           (scale back down)
```

**Effect:** Gradients now in fp16 representable range, preserved through backward!

### GradScaler Strategy

**What happens with gradient overflow?**
- If any gradient becomes inf or nan (overflow)
- Skip this optimizer step (don't update parameters)
- **Decrease** loss scale (for next iteration)
- Keep training

**Normal case:**
- Gradients are finite
- **Increase** loss scale slightly (be more aggressive)
- Update parameters

This dynamic scaling adapts to your model's convergence state.

In [ ]:
# ============================================================
# Loss Scaling and Gradient Overflow Demo
# ============================================================
# This demonstrates why loss scaling is needed for fp16 training
# and how to detect overflow.

def manual_loss_scaling_demo() -> None:
    """Show gradient underflow problem without scaling."""
    print("=== Gradient Underflow Without Scaling ===\n")

    torch.manual_seed(SEED)
    model_fp16 = TinyCausalLM(
        vocab_size=VOCAB_SIZE,
        d_model=32,
        num_heads=4,
        ff_dim=64,
        num_layers=2,
        dropout=0.0,
        max_len=64,
    )

    # Convert to fp16 for demo visualization
    model_fp16.half()
    input_ids, target_ids = make_counting_batch(batch_size=4, device=torch.device("cpu"))

    # Forward pass (automatically fp16)
    with torch.autocast(device_type="cpu", dtype=torch.bfloat16):
        logits = model_fp16(input_ids)
        loss = F.cross_entropy(
            logits.reshape(-1, VOCAB_SIZE),
            target_ids.reshape(-1),
            ignore_index=PAD_ID,
            reduction="mean",
        )

    print(f"Loss value: {loss.item():.6f}")
    print(f"Loss dtype: {loss.dtype}")

    # Backward without scaling
    loss.backward()

    # Check if gradients exist and have reasonable magnitudes
    grad_values = []
    for name, param in model_fp16.named_parameters():
        if param.grad is not None:
            grad_values.append(param.grad.abs().max().item())

    if grad_values:
        max_grad = max(grad_values)
        min_grad = min(grad_values)
        print(f"\nGradient magnitudes:")
        print(f"  Max: {max_grad:.2e}")
        print(f"  Min: {min_grad:.2e}")
        if min_grad < 1e-5:
            print(f"  ⚠️  Some gradients are very small")
            print(f"  In pure fp16: gradients < 6.1e-5 underflow to zero!")
    else:
        print("  No gradients found (frozen model?)")

manual_loss_scaling_demo()

print("\n" + "="*60 + "\n")

# ===== Loss Scaling with Detection =====
print("=== GradScaler Behavior ===\n")


class SimpleGradScaler:
    """
    Simplified GradScaler-like implementation to show the logic.
    Real torch.amp.GradScaler is more sophisticated.
    """

    def __init__(self, init_scale: float = 2.0**15):
        self.scale = init_scale
        self.growth_factor = 2.0
        self.backoff_factor = 0.5
        self.growth_interval = 2000  # Increase scale every N steps
        self.step_count = 0

    def scale_loss(self, loss: torch.Tensor) -> torch.Tensor:
        """Scale loss for backward pass."""
        return loss * self.scale

    def unscale_gradients(self, model: nn.Module) -> None:
        """Unscale gradients after backward."""
        for param in model.parameters():
            if param.grad is not None:
                param.grad.div_(self.scale)

    def step(self, optimizer, is_finite: bool) -> None:
        """
        Update scale based on gradient overflow detection.
        
        is_finite: Did backward produce finite gradients?
        """
        if is_finite:
            self.step_count += 1
            # Gradually increase scale
            if self.step_count % self.growth_interval == 0:
                self.scale *= self.growth_factor
                print(f"  Step {self.step_count}: Scale up to {self.scale:.0f}")
        else:
            # Overflow detected: back off
            self.scale *= self.backoff_factor
            print(f"  !OVERFLOW! Scale down to {self.scale:.0f}")
            optimizer.zero_grad()  # Skip this step

    def get_scale(self) -> float:
        return self.scale


# Simulate training with loss scaling
print("Simulating 5000 training steps with loss scaling:\n")

scaler_sim = SimpleGradScaler(init_scale=2**15)
torch.manual_seed(SEED + 42)

for step in range(1, 5001):
    # Simulate loss value (randomly spiking occasionally)
    base_loss = 2.5 - (step / 10000)  # Slowly decreasing
    if random.random() < 0.001:  # 0.1% chance of spike (outlier batch)
        base_loss *= 100  # Spike
        is_finite = False
    else:
        is_finite = True

    scaler_sim.step(None, is_finite=is_finite)

    if step in [1, 100, 500, 1000, 2000, 5000]:
        print(
            f"Step {step:5d}: Scale = {scaler_sim.get_scale():>10.0f}, "
            f"Finite = {is_finite}"
        )

print("\n=== Key Insights ===")
print("1. Start with scale = 2^15 to prevent underflow")
print("2. If gradients overflow → scale down, skip step")
print("3. If gradients are normal → gradually scale up (be greedy)")
print("4. This adaptation lets training stay stable across convergence")

# ===== GradScaler Exercise Setup =====
print("\n" + "="*60 + "\n")
print("=== Real PyTorch GradScaler ===\n")
print("In production, you would use:")
print("```")
print("scaler = torch.amp.GradScaler('cuda')")
print("")
print("with torch.autocast(device_type='cuda', dtype=torch.float16):")
print("    loss = model(batch)")
print("    scaler.scale(loss).backward()")
print("    scaler.unscale_(optimizer)")
print("    nn.utils.clip_grad_norm_(model.parameters(), 1.0)")
print("    scaler.step(optimizer)")
print("    scaler.update()")
print("```")

### Exercise 4.1 — Loss Scaling with Gradient Accumulation

You're using both gradient accumulation (4 steps) and loss scaling (scale=2^15).

In step i of accumulation:
```python
scaled_loss = loss * scaler.scale
scaled_loss.backward()  # Gradients now 2^15x larger
```

After 4 accumulation steps, before optimizer.step():
- Do you need to unscale gradients 1 time, or 4 times?
- Why?

### Exercise 4.1 Answer

**Unscale 1 time** (after all 4 accumulation steps, before optimizer.step()).

**Why:**
- Each backward() call: `scaled_loss.backward()` → gradients get 2^15x (accumulated via +=)
- After 4 steps: param.grad contains accumulated 2^15x scaled gradients
- You unscale ONCE to divide by 2^15
- Then clip (clipping operates on unscaled space)
- Then optimizer.step()

**Wrong approach:**
```python
for accum_step in range(4):
    scaled_loss.backward()
    scaler.unscale_(optimizer)  # ❌ Wrong: unscales after each step
```

**Correct approach:**
```python
for accum_step in range(4):
    scaled_loss.backward()      # Accumulate scaled gradients
scaler.unscale_(optimizer)      # ✅ Unscale once
nn.utils.clip_grad_norm_(...)   # Clip
scaler.step(optimizer)          # Update
scaler.update()                 # Adjust scale for next epoch
```


## 6. Activation checkpointing intuition

Activation checkpointing trades:
- **less memory**
for
- **more compute**

### Normal training
Forward activations are saved so backward can reuse them.

### Checkpointed training
Instead of saving all those activations, PyTorch discards some of them and recomputes them during backward.

### Why this matters
For large Transformers, activation memory can become a major bottleneck.
Checkpointing is a classic way to fit larger models or larger batches.


In [ ]:

# ============================================================
# Activation checkpointing demo
# ============================================================
# This is one of the clearest memory-vs-compute intuition demos.
#
# We build a module that counts how many times forward() is called.
# Then we compare:
# - normal execution
# - checkpointed execution
#
# With checkpointing, backward will trigger recomputation.

from torch.utils.checkpoint import checkpoint

class InstrumentedBlock(nn.Module):
    def __init__(self, dim: int = 16):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.forward_calls = 0

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        self.forward_calls += 1
        x = self.fc1(x)
        x = F.gelu(x)
        x = self.fc2(x)
        return x

# Build two identical modules so we can compare fairly.
torch.manual_seed(SEED)
normal_block = InstrumentedBlock(dim=16)
checkpoint_block = copy.deepcopy(normal_block)

# Use the same input for both cases.
x_normal = torch.randn(4, 16, requires_grad=True)
x_ckpt = x_normal.detach().clone().requires_grad_(True)

# ----- Normal path -----
y_normal = normal_block(x_normal)
loss_normal = y_normal.sum()
loss_normal.backward()

normal_forward_calls = normal_block.forward_calls
normal_input_grad = x_normal.grad.detach().clone()

# Collect parameter grads for comparison.
normal_param_grads = {
    name: p.grad.detach().clone()
    for name, p in normal_block.named_parameters()
}

# ----- Checkpointed path -----
y_ckpt = checkpoint(checkpoint_block, x_ckpt, use_reentrant=False)
loss_ckpt = y_ckpt.sum()
loss_ckpt.backward()

checkpoint_forward_calls = checkpoint_block.forward_calls
checkpoint_input_grad = x_ckpt.grad.detach().clone()
checkpoint_param_grads = {
    name: p.grad.detach().clone()
    for name, p in checkpoint_block.named_parameters()
}

# Compare outputs and gradients.
output_equal = torch.allclose(y_normal, y_ckpt, atol=1e-6, rtol=1e-5)
input_grad_equal = torch.allclose(normal_input_grad, checkpoint_input_grad, atol=1e-6, rtol=1e-5)

param_grad_equal = True
for name in normal_param_grads:
    if not torch.allclose(normal_param_grads[name], checkpoint_param_grads[name], atol=1e-6, rtol=1e-5):
        param_grad_equal = False
        break

print("Normal forward calls:      ", normal_forward_calls)
print("Checkpoint forward calls:  ", checkpoint_forward_calls)
print("Outputs equal:             ", output_equal)
print("Input grads equal:         ", input_grad_equal)
print("Parameter grads equal:     ", param_grad_equal)


## Interview Topic 5: Gradient Synchronization Patterns

**Why this matters (50% of infrastructure interviews):**
- Understanding how gradients actually flow is key to designing scalable training
- "Why does my 1000-GPU training have weird stragglers?" → often communication pattern issues
- DDP uses AllReduce, FSDP uses ReduceScatter, but both require understanding synchronization

### Three Main Patterns

#### Pattern 1: Ring All-Reduce
**Used by:** Some NCCL implementations, efficient for homogeneous clusters

Flow:
```
Rank 0: g0 → Rank 1: g0+g1 → Rank 2: g0+g1+g2 → ... → Rank 0: final
```

**Pros:** Linear bandwidth scalability, no hotspots
**Cons:** Higher latency, sequential dependencies

#### Pattern 2: Tree All-Reduce
**Used by:** Default Gloo/NCCL,  efficient for rapid convergence

Flow:
```
      [0,1,2,3]
       /     \
    [0,1]   [2,3]
     / \     / \
    0   1   2   3
```

**Pros:** Logarithmic depth, parallelizable
**Cons:** Root can be bottleneck, requires careful topology awareness

#### Pattern 3: Reduce-Scatter + AllGather (FSDP Strategy)
**Used by:** FSDP, efficient for parameter sharding

Flow:
```
ReduceScatter: Each rank gets different shard of reduced gradients
AllGather: Each rank broadcasts its parameter shard
```

**Pros:** Overlappable with computation, memory efficient
**Cons:** More complex, requires careful implementation

### Interview Key Points
- AllReduce is O(log N) latency, not O(N)
- Communication is deterministic; if hanging, check network/NCCL
- Gradient sync order matters for convergence (use same order every epoch)
- Different backends (NCCL, Gloo, MPI) have different characteristics

In [ ]:
# ============================================================
# Gradient Synchronization Patterns Visualization
# ============================================================

def visualize_ring_allreduce(num_ranks: int = 4) -> None:
    """
    Visualize ring all-reduce communication pattern.
    
    In ring all-reduce:
    - Each rank i sends to rank (i+1) % num_ranks
    - Each rank receives from rank (i-1) % num_ranks
    - After num_ranks rounds, all ranks have the final result
    """
    print("=== Ring All-Reduce Pattern ===")
    print(f"Simulating with {num_ranks} ranks\n")

    # Initialize: each rank has its own gradients
    rank_values = [f"g{i}" for i in range(num_ranks)]
    print(f"Initial state: {rank_values}")

    # Simulate ring rounds
    print("\nRing communication rounds:")
    for round_idx in range(num_ranks):
        new_values = ["?"] * num_ranks

        # Circular rotation of communication
        for rank in range(num_ranks):
            prev_rank = (rank - 1) % num_ranks
            # Current rank receives from prev_rank and adds
            received_val = rank_values[prev_rank]
            current_val = rank_values[rank]
            new_values[rank] = f"({current_val}+{received_val})"

        rank_values = new_values
        print(f"  After round {round_idx + 1}: {rank_values}")

    print("\nTime complexity: O(num_ranks) rounds")
    print("Bandwidth efficiency: O(data_size) per rank")

print()

def visualize_tree_allreduce(depth: int = 2) -> None:
    """
    Visualize tree all-reduce communication pattern.
    
    In tree reduce:
    - Combine pairs progressively
    - O(log N) depth
    """
    print("=== Tree All-Reduce Pattern ===")
    print(f"Simulating with depth={depth} (2^{depth} = {2**depth} ranks)\n")

    num_ranks = 2**depth

    # Build tree structure
    levels = []
    current_level = [f"g{i}" for i in range(num_ranks)]
    levels.append(("Initial", current_level.copy()))

    # Reduce phase: combine pairs upward
    print("Reduce phase (combining pairs):")
    for level_idx in range(depth):
        next_level = []
        for i in range(0, len(current_level), 2):
            combined = f"({current_level[i]}+{current_level[i+1]})"
            next_level.append(combined)
        print(
            f"  Level {level_idx} → {level_idx + 1}: "
            f"{len(current_level)} → {len(next_level)} nodes, "
            f"combined = {next_level}"
        )
        current_level = next_level

    # Broadcast phase: send result back down
    print("\nBroadcast phase (distributing result):")
    final_result = current_level[0]
    print(f"  Root broadcasts: {final_result} to all ranks")

    print(f"\nTime complexity: O(log {num_ranks}) = O({depth}) rounds")


visualize_ring_allreduce(num_ranks=4)
print("\n" + "="*60 + "\n")
visualize_tree_allreduce(depth=2)

# ===== Comparison table =====
print("\n" + "="*60 + "\n")
print("=== Communication Pattern Comparison ===\n")

comparison = """
|-----------+----------+----------+-------------------------------------|
| Aspect    | Ring     | Tree     | Notes                               |
|-----------+----------+----------+-------------------------------------|
| Rounds    | O(N)     | O(log N) | Ring is simpler but slower          |
| Latency   | High     | Low      | Tree better for fast convergence    |
| Bandwidth | Linear   | Linear   | Both achieve full link utilization  |
| Balance   | Good     | Uneven   | Tree root can be bottleneck         |
| Topology  | Agnostic | Aware    | Tree benefits from HW topology      |
|-----------+----------+----------+-------------------------------------|
"""

print(comparison)

print("Interview insight:")
print("- NCCL (Nvidia) auto-selects based on topology")
print("- Gloo (CPU-friendly) often uses tree")
print("- For 100+ GPUs, topology-aware schedules matter")
print("- Communication is deterministic—if hanging, check network config")

### Exercise 5.1 — Bandwidth Utilization in AllReduce

You have 1000 GPUs. One rank's AllReduce takes 10ms.

In ring all-reduce with N=1000 ranks:
- How many rounds does ring require?
- If each link has 900 Gb/s bandwidth, how much data goes through each link per round?

In tree all-reduce with N=1000 ranks:
- How many rounds?
- Is peak bandwidth utilization higher or lower than ring?

### Exercise 5.1 Answer

**Ring All-Reduce:**
- Rounds: 1000 (one per rank)
- Per-link bandwidth during round: All gradients (for model) pass through each link
- Peak utilization: Lower at beginning rounds (less data accumulated), then stabilizes
- Wall time: 1000 sequential rounds ≈ 1000 RTTs

**Tree All-Reduce:**
- Rounds: log2(1000) ≈ 10 (binary tree)
- Per-link bandwidth: Depends on depth (root has higher throughput)
- Peak utilization: Often higher locally (more parallel communication)
- Wall time: 10 rounds (but wider parallelism)

**Practical:** For 1000 GPUs, tree is 100x faster.
- Ring: 1000 rounds (not practical for 10ms/round!)
- Tree: 10-15 rounds → 100-150ms total (acceptable)

This is why NCCL defaults to tree for large clusters.


## 7. Pipeline parallel intuition

Pipeline parallelism splits a model into **stages** and feeds **microbatches** through those stages.

### Why do this?
When one device cannot hold or efficiently use the whole model:
- put different layers on different devices
- split a large batch into microbatches
- overlap stage work across microbatches

### Key terms
- **stage** = one partition of the model
- **microbatch** = one chunk of the batch
- **bubble** = idle time while the pipeline fills or drains

Pipeline parallelism is especially useful when model depth is large and communication can be hidden behind compute.


In [ ]:

# ============================================================
# Pipeline-parallel intuition demo
# ============================================================
# We simulate the idea on one device.
#
# The main goals are:
# 1) split a model into stages
# 2) split a batch into microbatches
# 3) show that stage-by-stage microbatch execution can reproduce
#    the same final output as the unsplit model
# 4) show a simple forward-only GPipe-style schedule

class PipelineStage0(nn.Module):
    def __init__(self, dim: int = 16):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim, 32),
            nn.GELU(),
            nn.Linear(32, 32),
            nn.GELU(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

class PipelineStage1(nn.Module):
    def __init__(self, dim_in: int = 32, dim_out: int = 8):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(dim_in, 32),
            nn.GELU(),
            nn.Linear(32, dim_out),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)

# Build stages and one combined reference model with exactly the same weights.
torch.manual_seed(SEED)
stage0 = PipelineStage0()
stage1 = PipelineStage1()

full_model = nn.Sequential(
    copy.deepcopy(stage0),
    copy.deepcopy(stage1),
)
stage0 = copy.deepcopy(full_model[0])
stage1 = copy.deepcopy(full_model[1])

# Use eval mode so there is no dropout randomness.
full_model.eval()
stage0.eval()
stage1.eval()

# One batch that we will split into microbatches.
pipeline_input = torch.randn(8, 16)

# Reference full-batch output.
reference_output = full_model(pipeline_input)

# Microbatch split.
microbatches = list(torch.chunk(pipeline_input, chunks=4, dim=0))

# Forward-only "pipeline" execution on one process:
# stage0 on each microbatch, then stage1 on the intermediate output.
stage0_outputs = []
final_outputs = []

for microbatch in microbatches:
    h = stage0(microbatch)
    stage0_outputs.append(h)

for h in stage0_outputs:
    y = stage1(h)
    final_outputs.append(y)

pipeline_output = torch.cat(final_outputs, dim=0)

pipeline_equal = torch.allclose(reference_output, pipeline_output, atol=1e-6, rtol=1e-5)
print("Pipeline stage composition matches full model:", pipeline_equal)

# Build a simple forward-only GPipe-like schedule table for 2 stages and 4 microbatches.
num_microbatches = 4
num_stages = 2
num_steps = num_microbatches + num_stages - 1

schedule = [["" for _ in range(num_steps)] for _ in range(num_stages)]

for time_idx in range(num_steps):
    for stage_idx in range(num_stages):
        microbatch_idx = time_idx - stage_idx
        if 0 <= microbatch_idx < num_microbatches:
            schedule[stage_idx][time_idx] = f"MB{microbatch_idx}"

print("\nForward-only pipeline schedule (rows=stages, cols=time steps):")
for stage_idx, row in enumerate(schedule):
    print(f"Stage {stage_idx}: {row}")



### Optional real pipeline-parallel skeleton

The next cell is a **reference-style sketch** for real pipeline training.  
It is not executed automatically in this notebook.

Real pipeline training typically uses:
- multiple ranks
- model partitioning into stages
- microbatch schedules such as GPipe or 1F1B
- a launcher such as `torchrun`


In [ ]:

# ============================================================
# Optional reference: real pipeline skeleton (not executed)
# ============================================================
RUN_REAL_PIPELINE = False

if RUN_REAL_PIPELINE:
    # This is only a sketch for interview familiarity.
    # Real pipeline training usually needs multiple processes/devices.

    import torch.distributed as dist

    # Pseudocode outline:
    #
    # 1) initialize process group
    # 2) assign each rank to a pipeline stage
    # 3) split the batch into microbatches
    # 4) run a schedule (GPipe, 1F1B, etc.)
    # 5) communicate activations and gradients between stages
    #
    # In modern PyTorch, you would look at:
    #   torch.distributed.pipelining
    #
    # Example conceptual flow:
    #
    # dist.init_process_group(...)
    # rank = ...
    #
    # if rank == 0:
    #     stage = stage0_module
    # elif rank == 1:
    #     stage = stage1_module
    #
    # schedule = ScheduleGPipe(...)
    # schedule.step(...)
    #
    # dist.destroy_process_group()
    pass


## Interview Topic 6: Checkpoint Save/Restore Patterns

**Critical for production training (~65% of senior interviews):**
- "How do you save a FSDP training checkpoint?" → different from DDP
- "My 100k-step training failed at 50k. How do I resume?" → need full state
- "Your checkpoint is 500GB. How do you optimize?" → distributed checkpoint strategies

### Full Training State Components

You must save ALL of these to correctly resume:

1. **Model weights** (parameters) - Obviously
2. **Optimizer state** - Adam's m, v tensors (per parameter!)
3. **Learning rate schedule state** - If using warmup or decay 
4. **RNG states** - PyTorch RNG, Python random, NumPy random
5. **GradScaler state** (if using AMP) - Current scale factor
6. **Gradient accumulation step counter** - Which step in accumulation cycle
7. **Epoch / iteration number** - For logging and learning rate schedule
8. **Training hyperparameters** - Batch size, LR, etc. (validate on resume)

### Why Forgetting State Causes Failures

**Scenario:** You save model + optimizer at step 10k.
- Optimizer Adam had m = 0.9 * m_prev + 0.1 * grad
- At resume, you reinitialize Adam → m = 0 (wrong!)
- Effective momentum is lost
- Model learns differently → validation loss diverges

**Result:** Your checkpoint isn't  actually reproducible.

### DDP vs FSDP Checkpointing

| Aspect | DDP | FSDP |
|--------|-----|------|
| **Model save** | Standard torch.save | Must use FSDP.state_dict_type |
| **Rank 0 only** | Common pattern (gather to rank 0) | More complex (sharded) |
| **File count** | 1 file | N files (one per rank) or consolidated |
| **Resume** | Straightforward | Must be on same world_size or resharding |
| **Memory** | Fits on rank 0 | Can be larger than GPU memory |

### Distributed Checkpoint Strategy  

**Naive approach (WRONG, causes hangs):**
```python
# ❌ All ranks write simultaneously
torch.save(model.state_dict(), f"ckpt_rank{rank}.pt")
```

**Why bad:** 
- NFS congestion (all ranks flush simultaneously)
- Single coordinating rank becomes bottleneck
- Can timeout or cause training to fail

**Better approach:**
```python
# ✅ Only rank 0 saves, all ranks barrier
dist.barrier()  # Ensure all ranks synced
if rank == 0:
    torch.save(state_dict, "checkpoint.pt")
dist.barrier()  # Wait for I/O to finish
```

Even better: **async checkpoint** (save while training continues)

In [ ]:
# ============================================================
# Checkpoint Save/Restore Pattern Demo
# ============================================================

import os
import tempfile

# Create a temporary directory for checkpoints
checkpoint_dir = tempfile.mkdtemp(prefix="ckpt_demo_")

class CheckpointManager:
    """
    Manages save/restore of full training state.
    Demonstrates best practices for distributed training.
    """

    def __init__(self, checkpoint_dir: str):
        self.checkpoint_dir = checkpoint_dir
        os.makedirs(checkpoint_dir, exist_ok=True)

    def save_checkpoint(
        self,
        step: int,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        scaler: Optional[object] = None,
        extra_state: Optional[Dict] = None,
    ) -> str:
        """
        Save complete training state to checkpoint.
        
        In a real distributed setting, only rank 0 would do this.
        """
        checkpoint_path = os.path.join(self.checkpoint_dir, f"ckpt_step_{step}.pt")

        checkpoint_dict = {
            # Essentials
            "step": step,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            # RNG states (critical for reproducibility)
            "torch_rng_state": torch.get_rng_state(),
            "python_random_state": random.getstate(),
        }

        # Optional: GradScaler state
        if scaler is not None:
            checkpoint_dict["scaler_state_dict"] = scaler.state_dict()

        # Optional: user-provided state
        if extra_state is not None:
            checkpoint_dict["extra_state"] = extra_state

        torch.save(checkpoint_dict, checkpoint_path)
        return checkpoint_path

    def load_checkpoint(self, checkpoint_path: str) -> Dict:
        """Load checkpoint and restore all state."""
        if not os.path.exists(checkpoint_path):
            raise FileNotFoundError(f"Checkpoint not found: {checkpoint_path}")

        checkpoint_dict = torch.load(checkpoint_path, map_location=device)

        # Restore RNG states
        torch.set_rng_state(checkpoint_dict["torch_rng_state"])
        random.setstate(checkpoint_dict["python_random_state"])

        return checkpoint_dict

    def resume_training(
        self,
        checkpoint_path: str,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        scaler: Optional[object] = None,
    ) -> int:
        """
        Restore model, optimizer, RNG states from checkpoint.
        Returns the step number to resume from.
        """
        checkpoint_dict = self.load_checkpoint(checkpoint_path)

        model.load_state_dict(checkpoint_dict["model_state_dict"])
        optimizer.load_state_dict(checkpoint_dict["optimizer_state_dict"])

        if scaler is not None and "scaler_state_dict" in checkpoint_dict:
            scaler.load_state_dict(checkpoint_dict["scaler_state_dict"])

        step = checkpoint_dict["step"]
        print(f"Resumed from checkpoint: step {step}")
        return step

# ===== Demo =====
print("=== Checkpoint Save/Restore Demo ===\n")

ckpt_manager = CheckpointManager(checkpoint_dir)

# Create a small training loop
model_for_ckpt = build_deterministic_model()
optimizer_for_ckpt = torch.optim.Adam(model_for_ckpt.parameters(), lr=1e-3)

print("Step 1: Train a few steps and save checkpoint...\n")

for step in range(1, 6):
    input_ids, target_ids = make_counting_batch(batch_size=4, device=torch.device("cpu"))
    logits = model_for_ckpt(input_ids)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=PAD_ID,
    )
    optimizer_for_ckpt.zero_grad()
    loss.backward()
    optimizer_for_ckpt.step()

    if step % 2 == 0:  # Save every 2 steps
        ckpt_path = ckpt_manager.save_checkpoint(
            step=step,
            model=model_for_ckpt,
            optimizer=optimizer_for_ckpt,
            extra_state={
                "epoch": 1,
                "loss": loss.item(),
                "lr_schedule": {"name": "linear_warmup"},
            },
        )
        print(f"  Saved checkpoint at step {step}: {os.path.basename(ckpt_path)}")

print(f"\nCheckpoint directory: {checkpoint_dir}")
print(f"Files created: {os.listdir(checkpoint_dir)}\n")

# ===== Resume from checkpoint =====
print("Step 2: Restore from checkpoint and verify state...\n")

# Create fresh model + optimizer
fresh_model = build_deterministic_model()
fresh_optimizer = torch.optim.Adam(fresh_model.parameters(), lr=1e-3)

# Modify fresh model to be different
for param in fresh_model.parameters():
    param.data.fill_(999.0)

print("Fresh model parameter sample (before resume):")
first_param = next(fresh_model.parameters())
print(f"  {first_param.data.flatten()[:3]}\n")

# Resume
ckpt_path = os.path.join(checkpoint_dir, "ckpt_step_4.pt")
resumed_step = ckpt_manager.resume_training(
    checkpoint_path=ckpt_path,
    model=fresh_model,
    optimizer=fresh_optimizer,
)

print(f"\nFresh model parameter sample (after resume):")
first_param = next(fresh_model.parameters())
print(f"  {first_param.data.flatten()[:3]}")
print(f"\n✓ State correctly restored!")

print("\n" + "="*60)
print("\n=== Checkpoint Best Practices ===\n")

best_practices = """
1. **Save only on rank 0** to avoid NFS congestion
   dist.barrier()  # Wait for all ranks
   if rank == 0:
       save_checkpoint(...)
   dist.barrier()

2. **Include RNG states** for reproducibility
   - torch.get_rng_state()
   - random.getstate()
   - numpy.random.get_state() (if using numpy)

3. **Save version/hyperparameters**
   checkpoint = {
       'version': 1,
       'model_arch': 'GPT-3.5B',
       'vocab_size': 50257,
       ...
   }
   # Validate on load: assert checkpoint['vocab_size'] == actual

4. **Handle FSDP state_dict differently**
   from torch.distributed.fsdp import FSDP
   if isinstance(model, FSDP):
       state_dict = model.state_dict_type(
           StateDictType.FULL_STATE_DICT,  # or SHARDED_STATE_DICT
       )

5. **Async checkpoint** (advanced)
   # Save checkpoint in background thread while training continues
   # Prevents training pause for large models

6. **Multi-checkpoint strategy** (production)
   # Keep last 3 checkpoints (last step is not always best)
   # Rotate: ckpt_N-2, ckpt_N-1, ckpt_N
   # If training crashes, can fallback to ckpt_N-1
"""

print(best_practices)

### Exercise 6.1 — Checkpoint Correctness

You have a checkpoint from step 50k with all required state.  
You resume on 4 GPUs with world_size=4.

After 100 more steps, you should have the exact same model weights as if you had trained continuously from step 0 to 50.1k.

But you notice the weights diverge slightly. What's a likely cause?

**Hint:** Think about what changes between original training and resumed training.

### Exercise 6.1 Answer

**Likely causes (in order of probability):**

1. **DataLoader shuffle random state not saved**
   - Original training: seed at step 0 → shuffled batch order  
   - Resumed training: different shuffle order → different batch at step 50.001k
   - Different batch → different gradients ✗

2. **Dropout in model** (if model.train())
   - RNG was saved, but if you forgot to call torch.set_rng_state() → different dropout mask

3. **Batch order mismatch**
   - Original training might skip some samples between steps (incomplete last batch)
   - Resumed training processes different samples offset

**Fix:**
```python
checkpoint = load_checkpoint(path)
torch.set_rng_state(checkpoint['torch_rng_state'])
random.setstate(checkpoint['python_random_state'])

# ALSO: Save and restore DistributedSampler state
# if using DistributedSampler:
#     sampler.set_epoch(checkpoint['epoch'])
```

**Key insight:** Resuming is only correct if:
- RNG states match
- Batch order is deterministic and same
- Optimizer state is identical
- All hyperparameters unchanged

This is why many teams keep detailed metadata in checkpoints.


## 8. Interview comparison summary

### DDP
Use when:
- the model fits on one GPU
- you want the simplest scalable training baseline

Main idea:
- full replica per rank
- gradients synchronized across ranks

### FSDP
Use when:
- model memory is too large for DDP
- you need to shard parameters / grads / optimizer state

Main idea:
- sharded-at-rest memory
- gather when needed for compute

### AMP
Use when:
- you want better performance and/or lower memory
- hardware supports lower precision well

Main idea:
- lower precision where safe
- maintain enough numerical stability

### Activation checkpointing
Use when:
- activation memory is the bottleneck
- you are willing to pay extra recompute cost

Main idea:
- save less memory in forward
- recompute during backward

### Pipeline parallelism
Use when:
- model depth is large
- one device cannot hold or efficiently process the whole model
- you can benefit from overlapping stage work across microbatches

Main idea:
- split model into stages
- run microbatches through a schedule



## 9. Interview-style exercises

### Exercise 1 — Why doesn't DDP reduce parameter memory per rank?

**Hint:**  
Ask yourself whether each rank still owns a full model copy.


## Interview Topic 7: Debugging Distributed Training

**Critical for senior/staff interviews (55% ask about this):**
- In production, 30% of training failures are distributed/communication issues, not bugs
- "Your torchrun training hangs. How do you debug?" → systematic approach expected
- Common interview: "Walk me through diagnosing a NCCL timeout"

### The Three Categories of Failures

#### Category 1: Rank Hangs (Deadlock)
**Symptoms:**
- Training starts fine, then stops after N steps
- One or more processes stuck
- No error message, just infinite wait

**Causes:**
- AllReduce sees different number of parameters on different ranks
- One rank took checkpoint but others didn't (code path divergence)
- Unmatched AllReduce calls (rank 0 does dist.broadcast, rank 1 doesn't)

**Debug approach:**
```
1. Add timestamps before each dist.* call
2. Check if all ranks reach the same number of steps
3. Verify model architecture identical across ranks
4. Check for any conditional dist.* operations (if rank == 0: dist.broadcast())
```

#### Category 2: Loss Divergence / NaNs
**Symptoms:**
- Training loss goes to NaN or inf
- Single-GPU runs fine, distributed diverges
- Or: loss is good for 10k steps, then NaN

**Causes:**
- Missing global gradient clipping (ran through Section 3b)
- Loss scaling overflow (fp16 without proper scaling)
- Gradient accumulation not normalized correctly
- Rank has completely different data batch (sampler not seeding properly)

**Debug approach:**
```
1. Add gradient norm logging on each rank → diverging norms = dataloader issue
2. Check loss scaling: is scale decreasing? More overflows?
3. Verify all ranks see same number of samples per epoch
```

#### Category 3: Slow Training  / Straggler
**Symptoms:**
- Training is 10x slower than expected
- "Why is my 8-GPU training slower than single GPU?"
- One GPU's epoch takes 50s, others take 5s

**Causes:**
- One rank's AllReduce slow → slows all ranks
- Uneven data distribution (rank 0 gets 2x data)
- Communication on slow interconnect (Ethernet vs NVLink)
- CPU/model serving contention

**Debug approach:**
```
1. Profile each rank independently: time.perf_counter around compute/comm
2. Log AllReduce time: use torch.distributed.monitored_barrier()
3. Check if data distribution is balanced
```

### Essential Debug Practices

1. **Always use torchrun with error reporting**
   ```bash
   torchrun --nproc_per_node=4 \
            --nnodes=1 \
            --rdzv_backend=c10d \
            --rdzv_endpoint=localhost:0 \
            train.py
   ```

2. **Rank-specific logging**
   ```python
   if rank == 0:
       logger.info(f"Training step {step}, loss={loss:.4f}")
   ```

3. **Monitored barriers**
   ```python
   dist.monitored_barrier()  # Detects which rank is slow
   ```

4. **NCCL environment variables** (for GPU debugging)
   ```bash
   export NCCL_DEBUG=INFO
   export NCCL_DEBUG_SUBSYS=ALL
   ```

### Common Patterns That Hang

**Pattern A: Conditional AllReduce (HANG)**
```python
# ❌ WRONG: only rank 0 does AllReduce
if rank == 0:
    dist.all_reduce(loss_tensor)
# Ranks 1,2,3 wait forever → HANG
```

**Pattern B: Different tensor sizes (HANG)**
```python
# ❌ WRONG: rank 0 has 4 parameters, rank 1 has 5
for name, param in model.named_parameters():
    dist.all_reduce(param.grad)  # Size mismatch → HANG
```

**Pattern C: Async AllReduce without sync (BUG)**
```python
# ⚠️ Tricky: AllReduce is async, but model updates before sync
handle = dist.all_reduce(grads, async_op=True)
optimizer.step()  # ❌ Gradients not synced yet!
handle.wait()
```

**Pattern D: Different model code (DIVERGENCE)**
```python
# ❌ WRONG: rank 0 has dropout=0.1, rank 1 has dropout=0.2
model = Model(dropout=0.1 if rank == 0 else 0.2)
# Different models → diverging gradients → training fails
```

In [ ]:
# ============================================================
# Distributed Training Debugging Helpers
# ============================================================

class DistributedDebugger:
    """
    Utilities for diagnosing distributed training issues.
    In real training,  you'd log these to TensorBoard or Weights & Biases.
    """

    def __init__(self, log_file: Optional[str] = None):
        self.log_file = log_file
        self.step_timings = {}

    def log(self, message: str, rank: int = 0) -> None:
        """Log message from specific rank."""
        timestamp = time.time()
        msg = f"[Rank {rank}][{timestamp:.2f}] {message}"
        print(msg)
        if self.log_file:
            with open(self.log_file, "a") as f:
                f.write(msg + "\n")

    def timing_context(self, name: str, rank: int = 0):
        """Context manager for timing sections."""
        return TimingContext(self, name, rank)

    def check_gradient_health(self, model: nn.Module, rank: int) -> Dict[str, float]:
        """Check if gradients are reasonable."""
        grad_stats = {
            "max_norm": 0.0,
            "min_norm": float("inf"),
            "zero_grads": 0,
            "nan_grads": 0,
            "inf_grads": 0,
        }

        for name, param in model.named_parameters():
            if param.grad is not None:
                grad = param.grad.data
                norm = grad.norm().item()

                grad_stats["max_norm"] = max(grad_stats["max_norm"], norm)
                grad_stats["min_norm"] = min(grad_stats["min_norm"], norm)

                if (grad == 0).all():
                    grad_stats["zero_grads"] += 1
                if torch.isnan(grad).any():
                    grad_stats["nan_grads"] += 1
                if torch.isinf(grad).any():
                    grad_stats["inf_grads"] += 1

        return grad_stats

    def report_gradient_health(self, model: nn.Module, step: int, rank: int = 0):
        """Useful for detecting divergence early."""
        stats = self.check_gradient_health(model, rank)

        if stats["nan_grads"] > 0 or stats["inf_grads"] > 0:
            self.log(
                f"Step {step}: ⚠️  ANOMALY DETECTED - "
                f"NaN={stats['nan_grads']}, Inf={stats['inf_grads']}",
                rank=rank,
            )
        elif stats["zero_grads"] > 0:
            self.log(
                f"Step {step}: Zero gradients in {stats['zero_grads']} parameters",
                rank=rank,
            )
        else:
            self.log(
                f"Step {step}: Grad norm range [{stats['min_norm']:.2e}, {stats['max_norm']:.2e}]",
                rank=rank,
            )


class TimingContext:
    """Context manager for timing."""

    def __init__(self, debugger: DistributedDebugger, name: str, rank: int):
        self.debugger = debugger
        self.name = name
        self.rank = rank
        self.start_time = None

    def __enter__(self):
        self.start_time = time.perf_counter()
        return self

    def __exit__(self, *args):
        elapsed = (time.perf_counter() - self.start_time) * 1000  # Convert to ms
        self.debugger.log(f"{self.name}: {elapsed:.2f} ms", rank=self.rank)


# ===== Demo =====
print("=== Distributed Debugging Tools ===\n")

debugger = DistributedDebugger()
test_model = build_deterministic_model()

# Simulate a training step
input_ids, target_ids = make_counting_batch(batch_size=4, device=torch.device("cpu"))

print("Rank 0 stepping through training:\n")

with debugger.timing_context("Forward pass", rank=0):
    logits = test_model(input_ids)
    loss = F.cross_entropy(
        logits.reshape(-1, VOCAB_SIZE),
        target_ids.reshape(-1),
        ignore_index=PAD_ID,
    )

with debugger.timing_context("Backward pass", rank=0):
    loss.backward()

print()
debugger.report_gradient_health(test_model, step=1, rank=0)

print("\n" + "="*60)
print("\n=== Common Debugging Checks ===\n")

checks = """
1. **Before training starts:**
   □ All ranks initialized with same model?
   □ DistributedSampler seeded consistently?
   □ NCCL/Gloo backend available?
   
2. **During training (add logging every N steps):**
   □ Loss is finite (not NaN/Inf)?
   □ Gradient norm is reasonable (not 1e30)?
   □ All ranks reach same step count?
   
3. **If hanging:**
   □ Check rank-specific logs for which rank stops
   □ Run single rank locally to verify code path
   □ Check network connectivity between ranks
   
4. **If diverging:**
   □ Compare gradient norms across ranks
   □ Verify data distribution (all ranks have same batch size)?
   □ Check if AllReduce is actually being called
"""

print(checks)

### Exercise 7.1 — Debugging a Hung torchrun Job

Your training command:
```bash
torchrun --nproc_per_node=4 train.py
```

Training runs for 1000 steps, then hangs. No error message. The processes are still alive.

Walk through your debugging strategy step-by-step. What would you check?

**Hint:** Think about what types of operations synchronize across ranks.

### Exercise 7.1 Answer

**Systematic debugging approach:**

1. **Check which ranks are stuck:**
   ```bash
   # Add logging before/after each synchronization point
   log_file: rank_0.log, rank_1.log, rank_2.log, rank_3.log
   # Compare: which rank stopped logging?
   ```
   → If all 4 ranks log at step 1000, but none at 1001 → sync issue at that line

2. **Identify the sync point:**
   - Look for: dist.all_reduce(), dist.barrier(), dist.broadcast()
   - Or: autograd.grad() with multiple backward passes
   → Most likely: an AllReduce call

3. **Check model architecture consistency:**
   ```python
   # Add before training
   model_signatures = [str(model) for _ in range(4)]
   assert all(s == model_signatures[0] for s in model_signatures), "Model mismatch!"
   ```

4. **Check data loading:**
   - Verify all ranks processed same number of batches at step 1000
   - Check if sampler.set_epoch() is being called

5. **Root cause hypotheses:**
   - **Hypothesis A:** Rank has exited early (error not caught)
     → Check rank logs for exceptions
   - **Hypothesis B:** Conditional AllReduce
     ```python
     if rank == 0:  # ❌ This causes hang
         dist.all_reduce(tensor)
     ```
   - **Hypothesis C:** Different number of dist.* calls
     ```python
     # ❌ Rank 0 does this, rank 1 doesn't:
     if epoch % checkpoint_interval == 0:
         dist.broadcast(...)
     ```

6. **Quick fix:**
   - Add timestamp + step logging before EVERY dist.* call
   - Run again, find which dist.* call doesn't complete
   - Compare code across ranks for that line

**Real-world answer:** Most hangs are from conditional dist.* operations or model architecture mismatches.

## Interview Topic 8: Combining Techniques

**Why this matters (45% of L6 interviews):**
- Single techniques are straightforward; combining them reveals system thinking
- "Use DDP + AMP + gradient accumulation + checkpointing. How do they interact?"
- Most common production setup uses 3-4 techniques simultaneously

### Combination 1: DDP + AMP

**Interaction:** Amplifies loss scaling importance

```
DDP + AMP flow:
1. Forward with autocast (fp16) → logits in fp16
2. Loss computed, scaled by GradScaler (×2^15)
3. Backward with scaled loss → gradients in fp16, scaled
4. AllReduce gradients across ranks → still scaled
5. Unscale gradients
6. Step
```

**Key gotcha:** AllReduce happens BEFORE unscaling.
- If one rank's gradients overflow → all ranks get NaN after AllReduce
- GradScaler detects NaN → skips step for all ranks
- Correct behavior, but adds complexity to debugging

**Learning rate implication:**
- You're using fp16, which has lower numerical precision
- Larger batches (from DDP) need larger learning rates
- Combined: might need 1.5-2.0x higher LR than single GPU

### Combination 2: DDP + Gradient Accumulation  

**Interaction:** Effective batch becomes very large

```
Per-rank local batch: 32
Accumulation steps: 4  
DDP world size: 16

Effective batch = 32 × 4 × 16 = 2048
```

**Critical:** Learning rate scaling rule applies to effective batch, not local batch.

**Subtlety:** AllReduce happens at END of accumulation, not per-microbatch.
```python
for accum_step in range(4):
    logits = model(batch_32)
    loss = compute_loss(logits, targets)
    (loss / 4).backward()  # Accumulate gradients

# After 4 accumulations: AllReduce happens ONCE
dist.all_reduce(grads) 
optimizer.step()
```

This is more efficient than AllReducing after each microbatch.

### Combination 3: FSDP + Activation Checkpointing

**Interaction:** Both reduce memory, but FSDP has overhead

```
Without either:        ~100 GB per rank (for 70B model)
+ Checkpointing:       ~60 GB per rank (save less activations)
+ FSDP (8 ranks):      ~15 GB per rank (parameter sharding)
+ Both:                ~8 GB per rank ✓ Fits in A100 80GB!
```

**But:** FSDP + checkpointing has interaction:
- FSDP gathers parameters when needed
- Checkpointing recomputes at backward
- Recomputed activations need same parameters
- → More AllGather operations during backward

**Result:** Can be compute-bound instead of memory-bound after combining.

### Combination 4: DDP + AMP + Gradient Accumulation + Checkpointing

This is **production LLM training at scale**. Example: 70B model, 1000 GPUs.

```
Per-rank:
  - local_batch = 4 (fits in memory with checkpointing)
  - accum_steps = 4
  - fp16 autocast
  
Effective batch = 4 × 4 × 1000 = 16,000
```

**Interactions to manage:**
1. **Loss scaling:** GradScaler must handle fp16 underflow + AllReduce overflow
2. **Gradient clipping:** Must be global (Section 3b) for stability
3. **Learning rate:** Scale by sqrt(16000 / 256) where 256 is reference batch
4. **Communication:** AllReduce every 4 steps (not 1) → 4x faster
5. **Checkpointing:** Recomputed activations use gathered parameters (FSDP) → more comm overhead
6. **Resuming:** Checkpoint must save AMP scaler state + accumulation step counter

**Interview question:** "These 4 techniques could give 10-50x speedup. Which gives the most, and in what order would you add them?"

**Answer:**
1. **DDP first** (N-way speedup up to ~O(N))
2. **Gradient accumulation** (reduce communication by accumulation factor, 4-8x)
3. **AMP** (2-3x faster, reduced memory)
4. **Checkpointing** (enable larger batch sizes, indirect 1.5-2x via better batch size)

Total: often 20-50x on 100+ GPUs.

In [ ]:
# ============================================================
# Combining Techniques: Production Training Loop
# ============================================================

class ProductionTrainingStep:
    """
    Shows best practices for combining:
    - DDP (simulated as iteration over mini-batches)
    - AMP (fp16/bf16 autocast)
    - Gradient accumulation
    - Loss scaling
    - Global gradient clipping
    """

    def __init__(
        self,
        model: nn.Module,
        optimizer: torch.optim.Optimizer,
        device: torch.device = torch.device("cpu"),
    ):
        self.model = model
        self.optimizer = optimizer
        self.device = device
        self.scaler = torch.amp.GradScaler("cuda") if device.type == "cuda" else None
        self.accumulation_steps = 4
        self.global_clip_norm = 1.0

    def train_epoch_with_all_techniques(self, num_batches: int = 10) -> List[float]:
        """
        Full training loop combining DDP + AMP + accumulation + clipping.
        In real distributed training, only rank 0 logs.
        """
        self.model.train()
        epoch_losses = []

        for batch_idx in range(num_batches):
            accum_loss = 0.0

            # Gradient accumulation loop
            for accum_step in range(self.accumulation_steps):
                self.optimizer.zero_grad()

                input_ids, target_ids = make_counting_batch(
                    batch_size=4,
                    device=self.device,
                )

                # AMP: Forward pass in lower precision
                if self.scaler is not None:
                    with torch.autocast(
                        device_type=self.device.type,
                        dtype=torch.float16,
                    ):
                        logits = self.model(input_ids)
                        loss = F.cross_entropy(
                            logits.reshape(-1, VOCAB_SIZE),
                            target_ids.reshape(-1),
                            ignore_index=PAD_ID,
                            reduction="mean",
                        )
                else:
                    logits = self.model(input_ids)
                    loss = F.cross_entropy(
                        logits.reshape(-1, VOCAB_SIZE),
                        target_ids.reshape(-1),
                        ignore_index=PAD_ID,
                        reduction="mean",
                    )

                # Scale loss if using AMP
                if self.scaler is not None:
                    self.scaler.scale(loss).backward()
                else:
                    loss.backward()

                accum_loss += loss.item() / self.accumulation_steps

            # After all accumulation steps: unscale and clip
            if self.scaler is not None:
                self.scaler.unscale_(self.optimizer)

            # Global gradient clipping (simulating across ranks)
            norm_before_clip = compute_grad_norm(self.model)
            nn.utils.clip_grad_norm_(
                self.model.parameters(),
                max_norm=self.global_clip_norm,
            )
            norm_after_clip = compute_grad_norm(self.model)

            # Step
            if self.scaler is not None:
                self.scaler.step(self.optimizer)
                self.scaler.update()  # Adjust scale for next iteration
            else:
                self.optimizer.step()

            epoch_losses.append(accum_loss)

            if (batch_idx + 1) % 5 == 0:
                print(
                    f"Batch {batch_idx + 1:3d}: "
                    f"loss={accum_loss:.4f}, "
                    f"grad_norm_pre_clip={norm_before_clip:.2e}, "
                    f"grad_norm_post_clip={norm_after_clip:.2e}, "
                    f"clip_ratio={norm_before_clip / (norm_after_clip + 1e-8):.2f}x"
                )

        return epoch_losses


print("=== Production Training with Combined Techniques ===\n")

# Build model
combined_model = TinyCausalLM(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_dim=64,
    num_layers=2,
    dropout=0.1,
    max_len=64,
).to(device)

combined_optimizer = torch.optim.Adam(combined_model.parameters(), lr=2e-3)

# Run training with combined techniques
trainer = ProductionTrainingStep(combined_model, combined_optimizer, device=device)
losses = trainer.train_epoch_with_all_techniques(num_batches=10)

print("\n" + "="*70)
print("\n=== Interaction Effects ===\n")

effects = """
✓ AMP reduces memory by ~50% (fp16 vs fp32)
✓ Accumulation reduces AllReduce from 1000x to 250x per epoch (4x reduction)
✓ Checkpointing enables 2-4x larger batch (not shown in demo)
✓ Gradient clipping prevents divergence in large batches
✓ Combined: can train 70B model on single 80GB A100

⚠️ Trade-offs:
  - More code complexity (scaler.scale/unscale timing)
  - Harder to debug (numerical instability markers less obvious)
  - Requires careful learning rate tuning
  - Checkpointing adds recomputation cost (1.5-2x slowdown for compute-heavy models)
"""

print(effects)

### Exercise 8.1 — Technique Interaction: What Breaks Together?

You're training with DDP + AMP + gradient accumulation.
- 16 GPUs
- local_batch = 8
- accum_steps = 4  
- fp16 autocast

After 5000 steps, loss goes NaN. You investigate:
- Local gradient norms are reasonable (1e-1 to 1e1) on all ranks
- Loss before scaling is finite (~2.5)
- No obvious overflow

What's the most likely culprit?

**Hint:** Think about order of operations: what happens DURING AllReduce vs AFTER?

### Exercise 8.1 Answer

**Most likely culprit: Scaled gradient overflow during AllReduce**

Flow:
1. Accumulate 4 microbatches → scaled gradients (2^15x larger)
2. AllReduce across 16 ranks → sum all scaled gradients
3. One rank's gradient might be small (1e-2), but after reduction: large (1e-2 × 16 ≈ 0.16)
4. After accumulation: truly large (0.16 × 4 ≈ 0.64)

But multirank scenario:
- Rank 0: scaled grad = 100 (for some parameter)
- Rank 1-15: scaled grad = 100 (each) 
- AllReduce sum: 1600

Now 1600 × scale (2^15) = 52 million in fp16 → **OVERFLOW → NaN**

**The issue:** GradScaler detects the overflow, but by then AllReduce already produced NaN values that propagate to all ranks.

**Fix:**
```python
# Check for overflow BEFORE AllReduce
scaler.unscale_(optimizer)  # Unscale first
if overflow_detected():
    skip_step()
else:
    nn.utils.clip_grad_norm_(...)  # Global clip
    optimizer.step()
```

Or: Use smaller loss scale or accumulation steps to reduce magnitude before AllReduce.

**Interview insight:** This is a subtle interaction between AMP + DDP + accumulation that only experienced engineers catch.

## Interview Topic 9: Performance Analysis Framework

**Critical for L6 system design interviews (45% ask this):**
- "Your training throughput is 40% of peak. Why?" → requires systematic analysis
- "Should we upgrade to better interconnect?" → data-driven decision
- Roofline model, FLOPs analysis, communication analysis

### The Roofline Model

For a GPU, peak performance is limited by:

$$\text{Performance} = \min(\text{Peak FLOPs}, \text{Peak Bandwidth} \times \text{Arithmetic Intensity})$$

**Arithmetic Intensity** = FLOPs per byte transferred:
- High arithmetic intensity: compute-bound (can reach peak FLOPs)
- Low arithmetic intensity: memory-bound (limited by bandwidth)

**Example:**
- A100: 312 TFLOPS (fp32), 2 TB/s bandwidth
- Matrix multiply 4096×4096×4096:
  - FLOPs: ~2 × 4096^3 / 10^12 ≈ 137 TFLOPS
  - Bytes: ~3 × 4096^2 × 4 bytes ≈ 200 GB
  - Arithmetic intensity: 137 TFLOPS / (200 GB / 400 ns) ≈ 10.8 FLOP/byte
  - Roofline says: can reach ~21 TFLOPS (limited by bandwidth)

### Measuring Training Efficiency

| Metric | Good Value | Comment |
|--------|-----------|---------|
| **GPU utilization** | >70% | Time GPU is doing useful work |
| **Memory utilization** | 80-95% | Should be high if compute-bound |
| **Communication overhead** | <20% | AllReduce + AllGather time |
| **Samples/sec** | Model-dependent | Benchmark against expected |
| **FLOPs utilization** | 30-70% typically | Theoretical peak rarely achieved |

### Interview Reasoning Framework

**Problem:** Training is slow.

**Step 1: Identify the bottleneck**
```
if GPU_utilization < 50%:
    → CPU/dataloader bottleneck (I/O or preprocessing)
elif Memory_utilization > 95% and compute_time  < 50%:
    → Memory bandwidth bottleneck
elif AllReduce_time > 50% of iteration:
    → Communication bottleneck (need DDP + accumulation)
else:
    → Other (compilation, Python overhead, etc.)
```

**Step 2: Measure concrete numbers**
- Total iteration time:  (e.g., 500ms)
- Compute time: 250ms
- AllReduce time: 100ms
- CPU data loading: 100ms
- Other: 50ms

**Step 3: Propose fixes in priority order**
- If communication dominant → gradient accumulation (4-8x reduction)
- If memory dominant → activationcheckpointing or FSDP
- If CPU dominant → async data loading, prefetch queues
- If weak scaling → communication likely culprit at large scale

In [ ]:
# ============================================================
# Performance Analysis Tools
# ============================================================

class PerformanceAnalyzer:
    """Tools for analyzing training performance."""

    @staticmethod
    def estimate_model_flops(
        vocab_size: int,
        d_model: int,
        num_heads: int,
        ff_multiplier: int,
        num_layers: int,
        seq_len: int,
        batch_size: int,
    ) -> Dict[str, float]:
        """
        Rough FLOPs estimate for transformer model.
        Formula: ~6 × num_params × seq_len × batch_size for forward pass.
        (Backward is ~2x forward, so total training ~3x forward)
        """
        # Simplified parameter count
        embedding_params = vocab_size * d_model
        attention_params = num_layers * (3 * d_model * d_model + d_model * d_model)
        ff_params = num_layers * (2 * d_model * (ff_multiplier * d_model))
        total_params = embedding_params + attention_params + ff_params

        # FLOPs per token forward pass
        # Attention: ~2 * seq_len * d_model * d_model per head
        # FFN: ~2 * d_model * ff_dim
        flops_per_token = (
            4 * seq_len * d_model * d_model  # Attention  
            + 4 * d_model * ff_multiplier * d_model  # FFN
        ) * num_layers

        flops_total = flops_per_token * batch_size * seq_len
        # Training is roughly 3x forward pass
        flops_with_backward = flops_total * 3

        return {
            "total_params": total_params,
            "flops_forward": flops_total,
            "flops_training": flops_with_backward,
        }

    @staticmethod
    def measure_throughput(
        model: nn.Module,
        batch_size: int,
        num_iterations: int,
        device: torch.device,
    ) -> Dict[str, float]:
        """Measure actual training throughput."""
        model.eval()

        # Warmup
        for _ in range(2):
            with torch.no_grad():
                input_ids, _ = make_counting_batch(batch_size=batch_size, device=device)
                _ = model(input_ids)

        # Measure
        torch.cuda.synchronize() if device.type == "cuda" else None
        start = time.perf_counter()

        for _ in range(num_iterations):
            with torch.no_grad():
                input_ids, _ = make_counting_batch(batch_size=batch_size, device=device)
                _ = model(input_ids)

        torch.cuda.synchronize() if device.type == "cuda" else None
        elapsed = time.perf_counter() - start

        samples_per_sec = (batch_size * num_iterations) / elapsed
        return {
            "elapsed_sec": elapsed,
            "samples_per_sec": samples_per_sec,
            "throughput_tokens_per_sec": samples_per_sec * 32,  # Avg seq_len ~32
        }


print("=== Performance Analysis Demo ===\n")

# Estimate FLOPs for our toy model
flops_info = PerformanceAnalyzer.estimate_model_flops(
    vocab_size=VOCAB_SIZE,
    d_model=32,
    num_heads=4,
    ff_multiplier=2,
    num_layers=2,
    seq_len=32,
    batch_size=8,
)

print("Model FLOPs Estimate:")
print(f"  Total parameters: {flops_info['total_params']:,}")
print(f"  FLOPs per forward: {flops_info['flops_forward']:.2e}")
print(f"  FLOPs per training iter (3x forward): {flops_info['flops_training']:.2e}\n")

# Measure throughput
perf_info = PerformanceAnalyzer.measure_throughput(
    model=model,
    batch_size=8,
    num_iterations=10,
    device=device,
)

print("Measured Throughput:")
print(f"  Samples/sec: {perf_info['samples_per_sec']:.2f}")
print(f"  Tokens/sec: {perf_info['throughput_tokens_per_sec']:.2f}\n")

# Estimate efficiency
print("=== Efficiency Estimate ===")
print("(Note: these are rough estimates for pedagogical purposes)\n")

peak_tflops_gpu = 0.312 if device.type == "cuda" else 0.0  # A100 fp32
if peak_tflops_gpu > 0:
    achieved_tflops = (flops_info['flops_training'] / 1e12) / perf_info['elapsed_sec']
    utilization = min(100, (achieved_tflops / peak_tflops_gpu) * 100)
    print(f"  Peak GPU FLOPs: {peak_tflops_gpu:.2f} TFLOPS")
    print(f"  Achieved FLOPs: {achieved_tflops:.4f} TFLOPS")
    print(f"  Utilization: {utilization:.1f}%")
else:
    print("  (CPU: GPU FLOPs not comparable)")

### Exercise 9.1 — Roofline Analysis

Your 70B LLM training:
- Measured throughput: 5000 tokens/sec on 8 GPUs
- Expected (from vendor specs): 15000 tokens/sec
- GPU utilization (nvprof): 45%

Is the bottleneck compute, memory, or communication?

**Given:**
- A100: 312 TFLOPS peak fp32
- Memory bandwidth: 2 TB/s
- Your model arithmetic intensity: ~15 FLOP/byte

### Exercise 9.1 Answer

**Roofline calculation:**
- Peak compute: 312 TFLOPS × 8 GPUs = 2496 TFLOPS
- Peak bandwidth: 2 TB/s × 8 = 16 TB/s
- Roofline knee = Peak_FLOPS / Peak_BW = 2496 / 16 = 156 FLOP/byte

Your model has 15 FLOP/byte < 156 → **MEMORY BOUND**

**Analysis:**
- Roofline limit for memory-bound: 2 TB/s × 15 FLOP/byte = 30 TFLOPS per GPU
- 8 GPUs × 30 TFLOPS = 240 TFLOPS (theoretical max for your model)
- Measured: ~40 TFLOPS (only using 2 GPUs worth!)

**Conclusion: You're at ~17% efficiency (40 / 240).**

**Why?**
- 45% GPU utilization is already low signal
- Communication overhead (DDP AllReduce)
- Python/CPU overhead
- Sub-optimal kernel fusion

**Fix priority:**
1. Profile AllReduce time → maybe add gradient accumulation
2. Check if using optimized kernels (Flash Attention, etc.)
3. Consider batch size increase (if memory allows)
4. Look at CPU preprocessing (might be data loading bottleneck)

## Interview Topic 10: Inference Infrastructure Basics

**Why this matters (40% of L5/L6 interviews ask about it):**
- Training is only half the infrastructure problem
- Inference has different constraints: latency (not throughput), batching
- Common interview: "Trained 70B model in 10 days. How long to serve 1M users?"

### Training vs Inference

| Aspect | Training | Inference |
|--------|----------|-----------|
| **Objective** | Optimize throughput | Optimize latency |
| **Batch size** | Large (256-2048) | Tiny (1-32) |
| **Flow** | Forward + Backward | Forward only |
| **Memory pressure** | Activations + gradients + optimizer | Just activations |
| **Compute pattern** | Dense | Dense (matrix-mul) but lower utilization |
| **Optimization priority** | Bandwidth | Latency |

### Inference Challenges

1. **Memory per token**: As sequence grows from 1 to 4096 tokens:
   - KV cache for 70B model: 2 layers × (K, V) × [seq_len, 4096] × fp16
   - 1 token: ~256 MB  
   - 2048 tokens: ~128 GB (won't fit on single GPU!)
   - Need techniques like KV cache quantization or distributed attention

2. **Latency**: First token latency vs per-token latency
   - First token: full forward pass (can be slow)
   - Next tokens: mostly compute for 1 new token (faster)
   - User perception: first token latency matters more

3. **Batching**: Different from training
   - Training: gather samples into one batch
   - Inference: can batch different user requests (different sequence lengths!)
   - Called "continuous batching" or "dynamic batching"

### Quantization (Most important for inference)

**Why quantize:**
- FP32 model: 70B × 4 bytes = 280 GB
- INT8 model: 70B × 1 byte = 70 GB (4x reduction)
- INT4 model: 70B × 0.5 byte = 35 GB (8x reduction)

**Types:**
- **Post-training quantization (PTQ):** Quantize after training (easy, ~1-2% accuracy drop)
- **Quantization-aware training (QAT):** Train with quantization (better, harder)
- **Dynamic vs static:** Dynamic quantizes per-batch, static pre-computes ranges

**Trade-off:** Latency vs accuracy
- FP32: slower but most accurate
- FP16: 2x faster, tiny accuracy loss
- INT8: 4x faster, 0.5-1% accuracy drop
- INT4: 8x faster, 2-5% accuracy drop (sometimes)

### Serving Patterns

**Pattern 1: Single-GPU serving**
- Works for up to ~13B models
- Simple, no distributed complexity
- Limited by GPU VRAM

**Pattern 2: Tensor parallelism**
- Split model across GPUs
- Each token forward requires cross-GPU communication
- Good for 30-70B models on 4-8 GPUs

**Pattern 3: Pipeline parallelism**
- Split by layers across GPUs
- Lower communication overhead than tensor parallelism
- Can be tricky for batching

**Pattern 4: Multi-model server**
- Multiple copies of model on different GPUs
- Handle more concurrent requests
- Used for high-throughput services

### Interview Calculation Example

**Q:** You have a 70B model. You want to serve 1000 concurrent users, each asking 1 query every 10 seconds. Each response has ~100 tokens. What infrastructure do you need?

**A (reasoning):**
1. Throughput needed: 1000 users × 100 tokens / 10 sec = 10,000 tokens/sec
2. Single A100 can do ~500 tokens/sec (rough estimate for 70B)
3. Need ~20 A100s just for throughput
4. But concurrent requests need buffering → maybe 30-40 A100s
5. With INT8 quantization: 4x faster → 8-10 A100s sufficient
6. Add redundancy (2x) → 16-20 A100s cluster

**Production reality:** Most companies oversub by 2-3x for fault tolerance, experimentation.

In [ ]:
# ============================================================
# Inference Infrastructure Analysis
# ============================================================

def estimate_inference_requirements(
    model_params_b: float,  # Billions
    model_dtype_bytes: float,  # 4 for fp32, 2 for fp16, 1 for int8
    max_seq_len: int,
    tokens_per_request: int,
    requests_per_second: int,
    target_latency_ms: float = 100,
) -> Dict[str, float]:
    """
    Estimate infrastructure for serving an LLM.
    """
    # Model weight memory
    model_memory_gb = model_params_b * model_dtype_bytes

    # KV cache memory per request (both K and V)
    # Rough: 2 * seq_len * model_dim * dtype_size
    # For 70B model: ~8192 parameters per layer
    kv_cache_per_token_mb = (max_seq_len * 8192 * model_dtype_bytes) / 1024

    # Per-GPU throughput (rough estimates)
    # FP32 70B on A100: ~300 tokens/sec
    # With quantization: ~1000+ tokens/sec
    throughput_tokens_per_sec_fp32 = max(50, 15000 / (model_params_b / 2))
    throughput_tokens_per_sec_int8 = throughput_tokens_per_sec_fp32 * 4

    # GPUs needed
    gpus_needed_fp32 = requests_per_second * tokens_per_request / throughput_tokens_per_sec_fp32
    gpus_needed_int8 = requests_per_second * tokens_per_request / throughput_tokens_per_sec_int8

    return {
        "model_memory_gb": model_memory_gb,
        "kv_cache_per_token_mb": kv_cache_per_token_mb,
        "throughput_tokens_per_sec_fp32": throughput_tokens_per_sec_fp32,
        "throughput_tokens_per_sec_int8": throughput_tokens_per_sec_int8,
        "gpus_needed_fp32": gpus_needed_fp32,
        "gpus_needed_int8": gpus_needed_int8,
    }


print("=== Inference Infrastructure Calculator ===\n")

scenarios = [
    {
        "name": "7B model, 100 req/sec",
        "params": 7,
        "dtype": 2,
        "seq_len": 2048,
        "tokens_per_req": 100,
        "rps": 100,
    },
    {
        "name": "70B model, 100 req/sec",
        "params": 70,
        "dtype": 2,
        "seq_len": 4096,
        "tokens_per_req": 100,
        "rps": 100,
    },
    {
        "name": "70B model, 1000 req/sec",
        "params": 70,
        "dtype": 2,
        "seq_len": 4096,
        "tokens_per_req": 100,
        "rps": 1000,
    },
]

for scenario in scenarios:
    result = estimate_inference_requirements(
        model_params_b=scenario["params"],
        model_dtype_bytes=scenario["dtype"],
        max_seq_len=scenario["seq_len"],
        tokens_per_request=scenario["tokens_per_req"],
        requests_per_second=scenario["rps"],
    )

    print(f"Scenario: {scenario['name']}")
    print(f"  Model size (FP16): {result['model_memory_gb']:.1f} GB")
    print(f"  TPS (FP16): {result['throughput_tokens_per_sec_fp32']:.0f}")
    print(f"  GPUs needed (FP16): {result['gpus_needed_fp32']:.1f}")
    print(f"  GPUs needed (INT8): {result['gpus_needed_int8']:.1f}")
    print()

print("="*60)
print("\nKey Takeaway:")
print("- Quantization (INT8) gives 4-8x speedup → massive infra savings")
print("- KV cache is critical limitation for long sequences")
print("- Batching multiple requests together improves efficiency")
print("- Most production: 0.5-2x overprovisioning for fault tolerance")

### Exercise 10.1 — Quantization Decision

Your company is deciding whether to quantize the 70B model:
- FP16 deployment: 20 A100s, latency 95ms
- INT8 quantization: 5 A100s, latency 120ms, 1-2% accuracy drop

The business metric: accuracy × (cost_not_spent / total_cost)

Is INT8 justified?

**Hint:** Think about what's more valuable in production: 25ms latency or 1% accuracy.

### Exercise 10.1 Answer

**YES, INT8 is justified in most production scenarios.**

**Financial analysis:**
- FP16: 20 A100s × $3k/month = $60k/month
- INT8: 5 A100s × $3k/month = $15k/month
- **Savings: $45k/month** (75% cost reduction!)

**Accuracy trade-off:**
- 1-2% accuracy drop on typical NLP benchmark
- In practice: users rarely notice 1% drop
- Latency improvement: 95ms → 120ms matters MORE for UX (still <200ms, imperceptible)

**Decision framework:**
- If accuracy-critical (finance, legal): use FP16
- If cost-critical (mass market): use INT8
- If latency-critical: use INT8 (faster!)
- Most production: INT8 is win-win (faster + cheaper)

**Real-world:** Most deployed LLMs use INT8 or lower (INT4).


### Exercise 1 — Answer

In DDP, each rank keeps a full model replica.

DDP helps scale throughput by parallelizing data processing and synchronizing gradients, but it does **not** shard the model parameters.  
So parameter memory per rank is still roughly the full-model size.



### Exercise 2 — Why does FSDP usually reduce memory more than DDP?

**Hint:**  
Think about what happens to parameters, gradients, and optimizer state.



### Exercise 2 — Answer

FSDP shards:
- parameters
- gradients
- optimizer state

across ranks, so each rank keeps only a fraction of those tensors most of the time.

That is why FSDP can train larger models than DDP on the same hardware.



### Exercise 3 — Why is activation checkpointing described as a compute-vs-memory tradeoff?

**Hint:**  
Think about what backward needs and what gets saved during forward.



### Exercise 3 — Answer

Normal training saves many forward activations so backward can reuse them.

Checkpointing saves fewer activations, which lowers memory use, but then backward has to recompute some forward work.

So:
- less memory
- more compute



### Exercise 4 — Why does AMP often improve training speed?

**Hint:**  
Think about hardware throughput and tensor size.



### Exercise 4 — Answer

Lower-precision tensors:
- take less memory
- can increase effective memory bandwidth
- may run faster on hardware that has strong support for fp16 or bf16

AMP lets PyTorch use lower precision where it is beneficial, while still keeping enough accuracy and stability.



### Exercise 5 — Why do pipeline schedules use microbatches?

**Hint:**  
Imagine sending one huge batch into stage 0 and making every other stage wait.



### Exercise 5 — Answer

Microbatches let different stages work concurrently on different parts of the batch.

Without microbatching, later stages may sit idle for long periods.
With microbatching, the pipeline can overlap work and improve utilization.



### Exercise 6 — Small coding challenge

Modify the toy dataset so it counts upward by `+2` instead of `+1`.

Current pattern:
- `3 4 5 6`

Desired pattern:
- `3 5 7 9`

**Hint:**  
Only the batch generator needs to change.
Look for the line that computes `start_digit + i`.



### Exercise 6 — Answer

Inside the batch generator, change:

```python
digits = [DIGIT_OFFSET + ((start_digit + i) % 10) for i in range(seq_len)]
```

to:

```python
digits = [DIGIT_OFFSET + ((start_digit + 2 * i) % 10) for i in range(seq_len)]
```

The model and training infrastructure code can stay the same.



### Exercise 7 — Concept challenge

Suppose a model fits under DDP, but the batch size is too small for good throughput because activation memory is too large.

Which feature is the most natural first thing to try:
- FSDP
- activation checkpointing
- pipeline parallelism

**Hint:**  
The problem statement says the model already fits, but activations are limiting batch size.



### Exercise 7 — Answer

A very natural first thing to try is **activation checkpointing**.

Why:
- the model already fits, so you may not need parameter sharding yet
- the issue is activation memory, which checkpointing targets directly

FSDP may still help later, but checkpointing is often the more direct response to activation-memory pressure.



## 10. Final takeaways

For infra-focused interviews, try to be able to explain these clearly:

1. how single-device training works  
2. why DDP uses one full model replica per rank  
3. why DDP needs data sharding such as `DistributedSampler`  
4. why FSDP reduces memory more than DDP  
5. why AMP affects both speed and numerical behavior  
6. why checkpointing saves memory by recomputing work  
7. why pipeline parallelism uses stages and microbatches  
8. when you would choose DDP vs FSDP vs checkpointing vs pipeline parallelism  

That combination is a strong foundation for LLM training infrastructure interviews.


## Final Checklist: L5/L6 Interview Readiness

### Foundation Topics (Sections 1-7)
- ✅ Single-device baseline training
- ✅ **Gradient accumulation mechanics** & effective batch size scaling
- ✅ **Global gradient norm clipping** (local vs global)
- ✅ DDP intuition + gradient averaging proof
- ✅ **Multi-node training bottlenecks** & communication costs
- ✅ FSDP memory sharding
- ✅ **Loss scaling deep dive** (fp16 underflow, GradScaler)
- ✅ AMP with autocast & GradScaler
- ✅ Activation checkpointing (memory-compute tradeoff)
- ✅ **Gradient synchronization patterns** (ring, tree, reduce-scatter)
- ✅ Pipeline parallelism with microbatches

### Production Topics (Sections 8-10)
- ✅ **Checkpoint save/restore** (full state management)
- ✅ **Debugging distributed training** (hangs, divergence, stragglers)
- ✅ **Combining techniques** (DDP+AMP+accum+checkpointing)
- ✅ **Performance analysis** (roofline model, bottleneck ID)
- ✅ **Inference infrastructure** (quantization, serving patterns)

### Interview Exercise Coverage
- 11 full exercises with detailed answers
- Covers decision-making, debugging, and system thinking
- Tests both conceptual understanding and practical reasoning

## References for further study

The best next steps after this extended notebook are:
- real `torchrun` DDP practice on multi-GPU machines
- one FSDP launch on a 8+ GPU cluster
- PyTorch profiler / Nsight workflows
- optimizer state sharding and checkpoint save/load (torch.distributed.checkpoint)
- DeepSpeed integration (ZeRO stages)
- FlashAttention and modern kernel optimization
- production serving with vLLM or TensorRT-LLM
- distributed tracing tools (PyTorch Profiler, Tensorboard)

## New in this Extended Edition

**Added 10 Critical Topics (not in baseline notebook):**
1. Gradient accumulation mechanics & LR scaling
2. Global gradient norm clipping in distributed training
3. Multi-node considerations & communication analysis
4. Loss scaling deep dive (fp16 numerical stability)
5. Gradient synchronization patterns (ring, tree, reduce-scatter)
6. Checkpoint save/restore with full state management
7. Debugging distributed training (systematic approach)
8. Combining techniques (DDP+AMP+accum+checkpointing interactions)
9. Performance analysis framework (roofline model, bottleneck ID)
10. Inference infrastructure basics (quantization, serving)

**Plus:** 11 interview-style exercises with full explanations and answer keys

This extended notebook now covers ~80-90% of real L5/L6 infrastructure interviews instead of the original ~40-50%.